# 15 · Modelado: Regresión logística (referencia interpretable) — mismo idioma que el gradient boosting

> **En BigQuery.** Lee `MUNICIPIO_ANIO` de `DATASET_DEFORESTACION` y, para comprobar que modela lo mismo, la muestra que guardó el gradient boosting en `DATASET_RESULTADOS`. Deja en `DATASET_RESULTADOS` ocho tablas de paso (`regresion_logistica_...`) con lo que compara el notebook 16, que las borra al terminar. **Qué tienes que cambiar:** nada; la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`). **Cuánto tarda:** unos diez minutos.
>
> Las tablas, gráficas y frases que calcula el notebook usan los años de la celda 1. Los textos escritos a mano (como este) describen la corrida del trabajo de grado, que entrena con 2021-2024 y prueba con 2025; si cambias los años, esos textos no se actualizan.

Entrena y evalúa la regresión logística regularizada sobre el panel `MUNICIPIO_ANIO`, **con la misma estructura, las
mismas variables, la misma muestra, la misma partición y las mismas métricas de `13_modelos_gradient_boosting`**. Es la
técnica de referencia de la Tabla 26: si el ensamble no le gana, su complejidad no se justifica, y es la que entrega
dirección, magnitud e incertidumbre de cada asociación (ODM-3, CDM-5). Las versiones
anteriores de este notebook se pueden consultar en el historial de git del repositorio.

Como en el gradient boosting, el panel anual predice dos cosas sobre la misma unidad, municipio-año:

1. **¿Deforesta?** (`tasa > 0`, clasificación) — regresión logística, con todas las filas.
2. **Dado que deforesta, ¿cuánto?** (la tasa del IDEAM, % del bosque) — se entrena y evalúa solo con los municipios-año
   con `tasa > 0`. La Tabla 26 descarta la regresión lineal "porque ajusta mal con tantos ceros y valores extremos";
   esa objeción se resuelve así: los ceros salen del problema por la estructura de dos partes, y para la magnitud
   (positiva y con cola pesada) se usa una **regresión Gamma con enlace logarítmico**, el modelo lineal generalizado
   estándar para tasas positivas asimétricas. Sus coeficientes se leen como efectos multiplicativos sobre la tasa.

**Qué se unificó con el gradient boosting (y por qué importa).**

| Elemento | Gradient boosting (referencia) | Este notebook |
|---|---|---|
| Muestra | 3.195 municipios-año rurales, 2021-2025, con bosque | **Idéntica** (se verifica fila a fila contra el notebook del gradient boosting) |
| Variables de entrada | 15 explicativas del **mismo año**, sin rezago (intensidades por km² y % del municipio) | **Las mismas 15**; el modelo usa las que quedan tras el VIF (2.2) |
| Partición | Entrena 2021-2024, prueba 2025 | Igual |
| Validación | `GroupKFold` por departamento (5 pliegues), en la búsqueda y en la prueba de territorios | Igual; los errores estándar también se agrupan por departamento |
| Selección de hiperparámetros | Exactitud balanceada (¿deforesta?) y R² (¿cuánto?) | Igual |
| Umbral | 0,5 | 0,5 |
| Métricas | Exactitud balanceada, AUC, precisión, recall, especificidad, F1, AP; R², RMSE, MAE, MedAE | Las mismas, más dos lecturas robustas para "¿cuánto?" (R² en escala log y Spearman) |
| Hiperparámetros | Tabla 27: `learning_rate`, `max_depth`... | Tabla 27, fila "Regresión logística": penalización L2 (y L1 de contraste); `C` ∈ {0,01; 0,03; 0,1; 0,3; 1; 10}; `class_weight` |

**Lo único específico de la logística** (y que el gradient boosting no necesita): una regresión logística supone que
el logit cambia linealmente con cada variable y que ninguna variable es casi combinación de las demás. Eso obliga a
(a) transformar con log(1+x) y estandarizar (2.1) y (b) podar la multicolinealidad con el VIF (2.2). Ambas decisiones
son las que ya fijaba la Tabla 26 y se hacen dentro de un `Pipeline`, así que se aprenden solo con el entrenamiento.

**Ampliaciones a la Tabla 27, con su razón.** Se agregan `C` = 0,03 y 0,3 (la grilla de la versión anterior mostró
que el óptimo con departamentos fuera está entre 0,03 y 0,3) y se deja que `class_weight` (ninguno o `balanced`) lo
decida la validación.

**Lo que se rescató de la versión anterior** (porque aporta algo que el gradient boosting no tiene):

- El VIF iterativo y la lectura de qué variables salen y por qué (2.2).
- La inferencia con **errores robustos agrupados por departamento** (contrastados con los agrupados por municipio), los
  *odds ratios* con intervalo de confianza, los **efectos marginales** y las **pruebas de Wald conjuntas** por bloque y
  por actividad (7.2).
- La comparación de especificaciones y la línea base de persistencia (5.4).
- La contribución de cada variable a la predicción de cada municipio, exacta en un modelo lineal (7.4).

**Lo que se agregó pensando en los objetivos.** La corrección de Holm por comparaciones múltiples y una sensibilidad con
efectos fijos de año (CDM-5 y la "estructura temporal" de ODM-3); la magnitud de "¿cuánto?" con su propia
inferencia; el cruce alto crédito–alto riesgo (OBJ-N1, ODM-2); una prueba explícita de reproducibilidad (CDM-6); y una
síntesis por objetivo **calculada** con los números reales (sección 9).

**Alcance.** Solo el panel anual municipio × año.

## 0. Preparación

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# Dataset donde quedan los resultados de la segmentación y de los
# modelos. Es el mismo de arriba. Para tenerlos aparte, pon otro
# nombre: el primer notebook que escriba ahí crea el dataset.
DATASET_RESULTADOS = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Años del modelo ---------------------------------------------------
# Se entrena con ANIO_INICIO a ANIO_PRUEBA - 1 y se prueba con ANIO_PRUEBA.
# Los del trabajo de grado son 2021 y 2025: entrena 2021-2024, prueba 2025.
# Para mover el modelo hacia adelante (por ejemplo, probar con 2026 cuando
# el panel municipal lo tenga completo), cambia ANIO_PRUEBA. Usa los mismos
# años en los notebooks 13 a 17.
ANIO_INICIO = 2021
ANIO_PRUEBA = 2025

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet scikit-learn==1.6.1 statsmodels==0.14.6 shap seaborn pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "15_modelos_regresion_logistica"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'panel_ideam' -> 'proyecto.dataset.PANEL_IDEAM'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA, PANEL_IDEAM...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    f"""Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= {ANIO_INICIO}".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None, procedencia=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "panel_ideam".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    procedencia    {"fuente": ..., ...}: si se da, deja una fila en la tabla
                   registro_cargas con qué se cargó, de dónde y cuándo.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

    if procedencia is not None:
        fila = pd.DataFrame([{
            "tabla": tabla.upper(),
            "filas": int(t.num_rows),
            "fuente": str(procedencia.get("fuente", "")),
            "parametros": json.dumps({k: v for k, v in procedencia.items() if k != "fuente"},
                                     ensure_ascii=False, default=str),
            "notebook": NOMBRE_NOTEBOOK,
            "cargado_en": datetime.now(timezone.utc),
        }])
        client.load_table_from_dataframe(
            fila, nombre_completo("registro_cargas", dataset),
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

In [ ]:
import json
import os
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from joblib import Parallel, delayed
from scipy import stats
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix,
                              f1_score, mean_absolute_error, mean_squared_error, median_absolute_error,
                              precision_recall_curve, precision_score, r2_score, recall_score,
                              roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid", font_scale=1.1, palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)


# En BigQuery solo quedan las tablas de datos. Los resultados se ven en el
# notebook; los únicos que se guardan son los que lee un notebook siguiente
# (tablas de paso). Las borra el último notebook que las usa (el 16 o el 17)
# y, si la cadena no se termina, vencen solas a los DIAS_TABLAS_DE_PASO días.
DIAS_TABLAS_DE_PASO = 7


def guardar_resultado(df, nombre, indice=False, descripcion=None):
    """Guarda en DATASET_RESULTADOS una tabla de paso para un notebook siguiente."""
    from datetime import timedelta
    if isinstance(df, pd.Series):
        df = df.to_frame()
    guardar_tabla(df, nombre, (descripcion or f"Resultado del notebook {NOMBRE_NOTEBOOK}: {nombre.replace('_', ' ')}.")
                  + " Tabla de paso: la borra el último notebook que la usa; si no, vence sola a los "
                  + f"{DIAS_TABLAS_DE_PASO} días.", dataset=DATASET_RESULTADOS, indice=indice)
    t = client.get_table(nombre_completo(nombre, DATASET_RESULTADOS))
    t.expires = datetime.now(timezone.utc) + timedelta(days=DIAS_TABLAS_DE_PASO)
    client.update_table(t, ["expires"])
    return df


SEMILLA = 0   # la misma del gradient boosting (CDM-6: reproducibilidad)
UMBRAL = 0.5  # el que usa .predict() del gradient boosting: todos los modelos clasifican con el mismo umbral

import statsmodels.api as sm
from sklearn.linear_model import GammaRegressor, LogisticRegression, TweedieRegressor
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

## 1. Cargar el panel y aplicar los filtros de 4.1

Mismos filtros que `09_eda_panel_municipal` y que el gradient boosting: municipios rurales o rurales
dispersos, según la clasificación del DNP que trae la tabla de municipios del Observatorio.

In [ ]:
RURALES = ["Rural", "Rural disperso"]

anual = leer_tabla("municipio_anio")
anual = anual[anual.ruralidad.isin(RURALES)].copy()

print(f"Filas: {len(anual):,}  ·  Municipios rurales: {anual.cod_dane.nunique():,}  ·  "
      f"Años: {sorted(int(a) for a in anual.anio.unique())}")
anual[["cod_dane", "anio", "nombre_municipio", "nombre_departamento", "ruralidad"]].head()

## 2. Las 16 variables del dataset final (15 explicativas + objetivo)

Se construyen **exactamente** como en el gradient boosting (crédito y ganado por km², cultivos y bosque como % del
municipio, área en km², todas del mismo año).

> **Un cambio respecto de la versión anterior.** Esa versión usaba montos en log(1+x) del año anterior, con
> 13 variables distintas tras el VIF (incluidas región y ruralidad). Eso la hacía responder una pregunta
> ligeramente distinta a la del gradient boosting y volvía incomparables los resultados. Aquí se alinea con el
> gradient boosting. La región y el rezago se conservan como sensibilidad en 5.4.

In [ ]:
km2 = anual.area_grilla_ha / 100
panel = anual.assign(
    tasa=100 * anual.tasa_def_ideam,
    cred=anual.credito_produccion_mm / km2,
    cred_gan=anual.credito_ganaderia_produccion_mm / km2,
    cred_palma=anual.credito_palma_produccion_mm / km2,
    cred_cacao=anual.credito_cacao_produccion_mm / km2,
    cred_cafe=anual.credito_cafe_produccion_mm / km2,
    cred_arroz=anual.credito_arroz_produccion_mm / km2,
    bovinos=anual.bovinos / km2,
    fincas=100 * anual.fincas_bovinas / km2,
    agricola=100 * anual.area_cosechada_ha / anual.area_grilla_ha,
    palma=100 * anual.cosechada_palma_ha / anual.area_grilla_ha,
    cacao=100 * anual.cosechada_cacao_ha / anual.area_grilla_ha,
    cafe=100 * anual.cosechada_cafe_ha / anual.area_grilla_ha,
    arroz=100 * anual.cosechada_arroz_ha / anual.area_grilla_ha,
    bosque=100 * anual.bosque_ideam_inicial_ha / anual.area_grilla_ha,
    area=km2,
).replace([np.inf, -np.inf], np.nan)

# Las 15 explicativas del dataset final (iguales a VARIABLES en 09_eda_panel_municipal y a
# EXPLICATIVAS en 13_modelos_gradient_boosting)
EXPLICATIVAS = ["cred", "cred_gan", "cred_palma", "cred_cacao", "cred_cafe", "cred_arroz", "bovinos",
                 "fincas", "agricola", "palma", "cacao", "cafe", "arroz", "bosque", "area"]

NOMBRES = {
    "cred": "Crédito de producción (M$/km²)", "cred_gan": "Crédito ganadero de producción (M$/km²)",
    "cred_palma": "Crédito de producción a palma (M$/km²)", "cred_cacao": "Crédito de producción a cacao (M$/km²)",
    "cred_cafe": "Crédito de producción a café (M$/km²)", "cred_arroz": "Crédito de producción a arroz (M$/km²)",
    "bovinos": "Bovinos por km²", "fincas": "Fincas ganaderas por 100 km²",
    "agricola": "Área cosechada (% del municipio)", "palma": "Palma (% del municipio)",
    "cacao": "Cacao (% del municipio)", "cafe": "Café (% del municipio)", "arroz": "Arroz (% del municipio)",
    "bosque": "Bosque (% del municipio)", "area": "Área del municipio (km²)",
}

columnas = ["cod_dane", "nombre_municipio", "anio", "nombre_departamento", "tasa", "def_ideam_ha"] + EXPLICATIVAS
panel = panel[columnas].sort_values(["cod_dane", "anio"]).copy()

# La EVA no tiene 2025: se completa con el ultimo cultivo conocido de cada municipio (2024) en vez de
# perder todas las filas de 2025 al hacer dropna (misma decision que el gradient boosting).
CULTIVOS = ["agricola", "palma", "cacao", "cafe", "arroz"]
panel[CULTIVOS] = panel.groupby("cod_dane")[CULTIVOS].ffill()

# El objetivo solo se evalua 2021-2025 (lo que tiene cifra del IDEAM); donde no hay bosque la tasa
# queda vacia (3.4) y se descarta.
muestra = panel[panel.anio.between(ANIO_INICIO, ANIO_PRUEBA) & panel.tasa.notna()].dropna(subset=EXPLICATIVAS).copy()
muestra["deforesta"] = (muestra.tasa > 0).astype(int)

print(f"Filas utilizables ({ANIO_INICIO}-{ANIO_PRUEBA}, con bosque): {len(muestra):,}")
print(f"Municipios-año con deforestación > 0: {muestra.deforesta.sum():,} "
      f"({100 * muestra.deforesta.mean():.1f} %)")
muestra[["cod_dane", "anio", "deforesta", "tasa"] + EXPLICATIVAS].head()

In [ ]:
# Verificación de "mismo idioma": la muestra tiene que ser la misma del gradient boosting (notebook 13), que
# la guardó en BigQuery: mismas filas y mismos valores en las 15 variables.
try:
    a = leer_tabla("gradient_boosting_muestra", dataset=DATASET_RESULTADOS)
except Exception:
    a = None
    print("Aviso: no está la tabla gradient_boosting_muestra (se crea al correr el notebook 13); "
          "no se pudo verificar que la muestra sea la misma.")
if a is not None:
    a = a.sort_values(["cod_dane", "anio"]).reset_index(drop=True)
    b = muestra.sort_values(["cod_dane", "anio"]).reset_index(drop=True)
    assert len(a) == len(b) and (a.cod_dane.astype(str) == b.cod_dane.astype(str)).all() and (a.anio == b.anio).all(), \
        "las filas no coinciden con las del gradient boosting: ¿usaste los mismos años y el mismo panel en el notebook 13?"
    assert np.allclose(a[EXPLICATIVAS].values.astype(float), b[EXPLICATIVAS].values.astype(float), equal_nan=True), \
        "los valores de las 15 variables no coinciden con los del gradient boosting"
    print(f"OK: mismas {len(b):,} filas y mismos valores en las 15 variables que el gradient boosting (notebook 13).")

# A partir de aqui el departamento es texto (evita categorias vacias al agrupar)
muestra["nombre_departamento"] = muestra.nombre_departamento.astype(str)

### 2.0 Partición temporal (se define aquí porque la transformación y el VIF se aprenden solo con el entrenamiento)

In [ ]:
train = muestra[muestra.anio.between(ANIO_INICIO, ANIO_PRUEBA - 1)].reset_index(drop=True)
test = muestra[muestra.anio == ANIO_PRUEBA].reset_index(drop=True)

X_train, y_train_clas, y_train_tasa = train[EXPLICATIVAS], train.deforesta, train.tasa
X_test, y_test_clas, y_test_tasa = test[EXPLICATIVAS], test.deforesta, test.tasa
grupos_train = train.nombre_departamento

# Mismos pliegues que el gradient boosting: GroupKFold por departamento, 5 pliegues, es determinista
N_PLIEGUES = 5

print(f"Entrena ({ANIO_INICIO}-{ANIO_PRUEBA - 1}): {len(train):,} filas, {grupos_train.nunique()} departamentos")
print(f"Prueba ({ANIO_PRUEBA}): {len(test):,} filas")
pd.DataFrame({"filas": [len(train), len(test)], "no deforesta": [(y_train_clas == 0).sum(), (y_test_clas == 0).sum()],
              "deforesta": [(y_train_clas == 1).sum(), (y_test_clas == 1).sum()],
              "% deforesta": [100 * y_train_clas.mean(), 100 * y_test_clas.mean()]},
             index=[f"entrenamiento {ANIO_INICIO}-{ANIO_PRUEBA - 1}", f"prueba {ANIO_PRUEBA}"]).round(1)

In [ ]:
asimetria = pd.DataFrame({"asimetría original": X_train.skew(), "asimetría tras log(1+x)": np.log1p(X_train).skew()})
asimetria.index = [NOMBRES[v] for v in asimetria.index]
print("Asimetría (0 = simétrica), en el entrenamiento, antes y después de log(1+x):")
asimetria.round(2)

### 2.1 Qué necesita la logística que el gradient boosting no necesita

**Transformación logarítmica.** Las intensidades son muy asimétricas (la mayoría de los municipios tiene crédito a palma
igual a cero y unos pocos tienen valores enormes) y la logística supone que el logit cambia linealmente con cada
variable. `log(1+x)` conserva los ceros, comprime la cola y hace razonable esa linealidad. En los árboles es
inofensivo; aquí es necesario. No se borra ni se recorta ningún valor: los atípicos "son los municipios que el negocio
busca" (OBJ-N1), solo se llevan a una escala donde un solo municipio no domina el ajuste.

**Estandarización.** La regularización (L1 o L2) penaliza el tamaño de los coeficientes; si las variables tienen escalas
distintas, unas se penalizan más que otras. Además, estandarizadas, los *odds ratios* se comparan entre variables:
cada uno es el efecto de **+1 desviación estándar** (en la escala logarítmica).

Todo vive dentro de un `Pipeline` que recibe las **15 columnas** del gradient boosting y selecciona internamente las que
el modelo usa. Así los tres modelos del equipo reciben la misma tabla de entrada, y la permutación por bloques
(7.5) es idéntica en los tres.

In [ ]:
def preprocesador(columnas):
    # log(1+x) y estandarizacion solo sobre `columnas`; el resto de las 15 se descarta dentro del Pipeline
    return ColumnTransformer([
        ("num", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                          ("esc", StandardScaler())]), list(columnas)),
    ], remainder="drop", verbose_feature_names_out=False)


def pipe_lineal(modelo, columnas):
    return Pipeline([("pre", preprocesador(columnas)), ("m", modelo)])

### 2.2 Multicolinealidad: VIF iterativo

Con colinealidad fuerte los coeficientes se vuelven inestables, sus errores estándar se inflan y pueden cambiar de
signo. Aquí hay colinealidad por construcción: el crédito total es la suma de los créditos por cadena, el área
cosechada total incluye a cada cultivo, y el crédito de una cadena casi duplica al área de esa cadena. El **factor de
inflación de varianza (VIF)** lo mide: se calcula sobre el entrenamiento; si la variable con mayor VIF supera **5**,
se elimina y se recalcula, hasta que todas queden por debajo. Es el criterio de la Tabla 26. No cambia la
información que entra al sistema (las 15 variables siguen siendo las del gradient boosting): decide cuáles puede
sostener una regresión lineal con coeficientes legibles.

In [ ]:
def calcular_vif(columnas):
    Z = pd.DataFrame(StandardScaler().fit_transform(np.log1p(X_train[columnas])), columns=columnas)
    Zc = sm.add_constant(Z)
    return pd.Series([variance_inflation_factor(Zc.values, i) for i in range(1, Zc.shape[1])], index=columnas)


UMBRAL_VIF = 5
VARS_LR = list(EXPLICATIVAS)
historial, eliminadas = [], []
while True:
    v = calcular_vif(VARS_LR)
    historial.append(v.rename(f"ronda {len(historial) + 1}"))
    if v.max() <= UMBRAL_VIF:
        break
    sale = v.idxmax()
    print(f"Ronda {len(historial)}: se elimina {NOMBRES[sale]} (VIF = {v.max():.2f})")
    eliminadas.append((sale, round(v.max(), 2)))
    VARS_LR.remove(sale)

print(f"\nVariables eliminadas: {[NOMBRES[e] for e, _ in eliminadas] or 'ninguna'}  ·  VIF máximo final: {historial[-1].max():.2f}")
print(f"El modelo usa {len(VARS_LR)} de las 15 variables.")
vif_tabla = pd.concat(historial, axis=1)
vif_tabla.index = [NOMBRES[i] for i in vif_tabla.index]
vif_tabla.round(2)

In [ ]:
# Que representa a cada variable eliminada: con cual de las que se quedaron esta mas correlacionada
Zlog = np.log1p(X_train)
for sale, _ in eliminadas:
    corr = Zlog[VARS_LR].corrwith(Zlog[sale], method="spearman").sort_values(key=abs, ascending=False)
    print(f"{NOMBRES[sale]} queda representada por: {NOMBRES[corr.index[0]]} (rho de Spearman = {corr.iloc[0]:+.2f})")
ESTRUCTURALES_LR = [v for v in VARS_LR if v in ("bosque", "area")]

## 3. Partición: 2021-2024 para entrenar, 2025 para probar (definida en 2.0)

Igual que el gradient boosting: se entrena con 2021 a 2024 y se prueba con 2025, sin tocar 2025 durante la búsqueda de
hiperparámetros. La partición se construyó en la sección 2.0 porque la transformación y el VIF se aprenden solo con el entrenamiento. El departamento se guarda aparte para la validación por departamentos completos (sección 4 y
sección 6) y para agrupar los errores estándar (sección 7).

## 4. Búsqueda de hiperparámetros (Tabla 27)

**¿Deforesta?** Regresión logística con `penalty` ∈ {L2, L1}, `C` ∈ {0,01; 0,03; 0,1; 0,3; 1; 10} y `class_weight` ∈
{ninguno, `balanced`}: 24 combinaciones. La técnica de búsqueda es una **grilla exhaustiva** (*grid search*): se evalúan
todas las combinaciones, sin muestreo aleatorio ni optimización bayesiana, igual que en gradient boosting y random forest.

- **`penalty` y `C`.** `C` es el inverso de la fuerza de regularización (menor = coeficientes más encogidos). L2 reparte
  el peso entre variables correlacionadas; L1 puede llevar coeficientes a cero, es decir, hace selección de variables. Con
  pocos municipios y departamentos que se parecen internamente, una regularización fuerte suele generalizar mejor
  que una débil, y eso es justo lo que la validación por departamento mide.
- **`class_weight`.** Compensa que haya más municipios-año que deforestan (63 %) que los que no. Mejora la exactitud
  balanceada, pero **desplaza las probabilidades** (las vuelve menos calibradas); por eso se deja que la validación
  decida y se mira la curva de calibración en 5.1.

**¿Cuánto?** Regresión Gamma (enlace logarítmico) con penalización L2, `alpha` ∈ {0,001; 0,01; 0,1; 1; 10; 100}
(equivalente a `C`: mayor `alpha` = más regularización), solo sobre las filas con `deforesta == 1`.

**Cómo se valida cada combinación.** `GroupKFold` por departamento (5 pliegues) sobre 2021-2024, con los mismos
pliegues del gradient boosting y de `random_forest`; **2025 no se toca**. Se selecciona por exactitud balanceada
(¿deforesta?) y R² (¿cuánto?), como el gradient boosting, y se registran el AUC y la desviación entre pliegues.

In [ ]:
from itertools import product

GRID_CLAS = [dict(penalty=p, C=C, class_weight=w)
             for p, C, w in product(("l2", "l1"), (0.01, 0.03, 0.1, 0.3, 1, 10), (None, "balanced"))]
GRID_REG = [dict(alpha=a) for a in (0.001, 0.01, 0.1, 1, 10, 100)]


def modelo_clas(params):
    return pipe_lineal(LogisticRegression(solver="liblinear", max_iter=5000, random_state=SEMILLA, **params), VARS_LR)


def modelo_reg(params):
    return pipe_lineal(GammaRegressor(max_iter=5000, **params), VARS_LR)


def buscar(construir, X, y, grupos, grid, tipo):
    # Recorre `grid`, evalua cada combinacion con GroupKFold por departamento (los mismos pliegues del gradient
    # boosting) y devuelve (id de la mejor combinacion, tabla de resultados).
    pliegues = list(GroupKFold(n_splits=N_PLIEGUES).split(X, y, groups=grupos))
    filas = []
    for k, params in enumerate(grid):
        rs = []
        for i_train, i_val in pliegues:
            mod = construir(params).fit(X.iloc[i_train], y.iloc[i_train])
            yv = y.iloc[i_val]
            if tipo == "clas":
                prob = mod.predict_proba(X.iloc[i_val])[:, 1]
                rs.append(dict(metrica=balanced_accuracy_score(yv, prob >= UMBRAL), auc=roc_auc_score(yv, prob),
                               ap=average_precision_score(yv, prob)))
            else:
                pred = mod.predict(X.iloc[i_val])
                rs.append(dict(metrica=r2_score(yv, pred), mae=mean_absolute_error(yv, pred),
                               r2_log=r2_score(np.log(yv), np.log(np.clip(pred, 1e-4, None)))))
        rs = pd.DataFrame(rs)
        filas.append({"id": k, **{h: str(v) if h == "class_weight" else v for h, v in params.items()},
                      **rs.mean().to_dict(), "metrica_sd": rs.metrica.std()})
    tabla = pd.DataFrame(filas).sort_values("metrica", ascending=False, kind="stable").reset_index(drop=True)
    return int(tabla.loc[0, "id"]), tabla


def describir(params):
    return ", ".join(f"{k}={v}" for k, v in params.items())

In [ ]:
# Clasificacion: ¿deforesta? -> se maximiza la exactitud balanceada
id_clas, tabla_clas = buscar(modelo_clas, X_train, y_train_clas, grupos_train, GRID_CLAS, "clas")
params_clas = GRID_CLAS[id_clas]
print("Mejor combinación (clasificación):", describir(params_clas))
tabla_clas.head(8).round(4)

In [ ]:
# Regresion: dado que deforesta, ¿cuanto? -> solo sobre las filas con deforesta == 1, se maximiza R2
en_deforesta = (y_train_clas == 1).values
id_reg, tabla_reg = buscar(modelo_reg, X_train[en_deforesta], y_train_tasa[en_deforesta], grupos_train[en_deforesta],
                           GRID_REG, "reg")
params_reg = GRID_REG[id_reg]
print(f"Filas de entrenamiento con deforesta == 1: {en_deforesta.sum():,} de {len(X_train):,}")
print("Mejor combinación (regresión):", describir(params_reg))
tabla_reg.round(4)

### 4.1 Cómo se ve la búsqueda, y qué cambia al agrupar por departamento

La figura muestra la exactitud balanceada de validación para cada `C`, con L1 y L2 y con y sin pesos. Es la forma
directa de ver qué tan plana es la superficie: si casi todas las combinaciones dan lo mismo, el modelo es estable frente
a los hiperparámetros y la elección no es frágil. Debajo, la combinación ganadora se valida también agrupando por
**municipio** (la validación de la versión original): la diferencia es el costo de ser honestos con la
autocorrelación espacial.

In [ ]:
t = tabla_clas.copy()
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for (pen, cw), d in t.groupby(["penalty", "class_weight"]):
    d = d.sort_values("C")
    for eje, col, nombre in [(ax[0], "metrica", "Exactitud balanceada"), (ax[1], "auc", "AUC")]:
        eje.errorbar(d.C.astype(float), d[col], yerr=d.metrica_sd if col == "metrica" else None, marker="o", capsize=3,
                     label=f"{pen}, {cw}")
        eje.set(xscale="log", xlabel="C (escala log)", ylabel=f"{nombre} en validación", title=f"{nombre} (departamentos fuera)")
ax[0].legend(title="penalty, class_weight", fontsize=9)
plt.tight_layout()
plt.show()

def validar_agrupando_por(grupos):
    r = []
    for i, v in GroupKFold(n_splits=N_PLIEGUES).split(X_train, y_train_clas, groups=grupos):
        prob = modelo_clas(params_clas).fit(X_train.iloc[i], y_train_clas.iloc[i]).predict_proba(X_train.iloc[v])[:, 1]
        r.append((balanced_accuracy_score(y_train_clas.iloc[v], prob >= UMBRAL), roc_auc_score(y_train_clas.iloc[v], prob)))
    return np.mean(r, axis=0)

mejor_fila = tabla_clas.loc[0]
ba_mun, auc_mun = validar_agrupando_por(train.cod_dane)
print(f"Misma combinación ganadora, validada de dos maneras ({ANIO_INICIO}-{ANIO_PRUEBA - 1}):")
print(f"  agrupando por DEPARTAMENTO: exactitud balanceada {mejor_fila.metrica:.3f}  ·  AUC {mejor_fila.auc:.3f}")
print(f"  agrupando por MUNICIPIO:    exactitud balanceada {ba_mun:.3f}  ·  AUC {auc_mun:.3f}")
print(f"Costo de agrupar por departamento: {auc_mun - mejor_fila.auc:+.3f} de AUC.")

**Lectura: por qué la elección de hiperparámetros es sólida (CDM-7, REQ-04).**

- **La superficie es casi plana.** Las ocho mejores combinaciones de ¿deforesta? quedan entre 0,809 y 0,815 de exactitud balanceada, todas con `class_weight = balanced`, y la
  desviación entre pliegues (≈ 0,03) es mayor que la diferencia entre ellas. La ganadora (L1, `C` = 0,03) no es "la única buena": otra combinación daría el mismo resultado práctico. Por eso
  la elección no es frágil y se justifica por otra razón, no por una décima de métrica.
- **Esa razón es la interpretabilidad.** Con 639 municipios y departamentos que se parecen entre sí, un modelo con coeficientes encogidos y que descarta las variables sin señal
  (L1 llevó a cero cuatro: crédito a palma, crédito a arroz, área cosechada y bovinos) es más fácil de explicar a un interesado (REQ-04) y no cuesta desempeño.
- **El costo de ser honestos.** Agrupar por departamento en vez de por municipio resta 0,009 de AUC. Esa diferencia es lo que se paga por no dejar que el modelo "vea" departamentos vecinos
  durante la validación; es la cifra que respalda que el desempeño reportado después es el de un territorio nuevo.

## 5. Modelo final y evaluación en 2025

Se reentrena cada modelo sobre todo 2021-2024 con la combinación ganadora y se evalúa una sola vez sobre 2025, que no
se tocó antes. Además de las métricas de la matriz de confusión y del AUC, se mide la brecha entrenamiento-prueba contra el
15 % de CDM-7. A diferencia de un bosque, una regresión regularizada casi no memoriza, así que la brecha medida contra
el entrenamiento dentro de la muestra es una medida razonable; se reporta también contra la validación por
departamentos de la búsqueda (la definición literal de CDM-7: "validación cruzada con bloques geográficos").

In [ ]:
def metricas_clasificacion(y, pred, prob, etiqueta):
    # Las mismas del gradient boosting: precision, recall/sensibilidad, especificidad, F1, AUC y AP.
    precision = precision_score(y, pred, zero_division=0)
    recall = recall_score(y, pred)
    especificidad = recall_score(y, pred, pos_label=0)
    f1 = f1_score(y, pred)
    auc = roc_auc_score(y, prob)
    ap = average_precision_score(y, prob)
    print(f"[{etiqueta}] precision: {precision:.3f}  ·  recall/sensibilidad: {recall:.3f}  ·  "
          f"especificidad: {especificidad:.3f}  ·  F1: {f1:.3f}  ·  AUC: {auc:.3f}  ·  "
          f"precisión promedio (AP): {ap:.3f}")
    return dict(precision=precision, recall=recall, especificidad=especificidad, f1=f1, auc=auc, ap=ap)


def metricas_regresion(y, pred, etiqueta):
    # R², RMSE, MAE y MedAE como en el gradient boosting. Se agregan dos lecturas que no dependen de unos
    # pocos municipios extremos (5.3 del gradient boosting): el R² en escala logaritmica y la correlacion
    # de rangos de Spearman entre la tasa real y la predicha. Se calculan solo a partir de las
    # predicciones, asi que se pueden aplicar igual a cualquier modelo (incluido el gradient boosting).
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    rmse = mean_squared_error(y, pred) ** 0.5
    mae = mean_absolute_error(y, pred)
    medae = median_absolute_error(y, pred)
    r2 = r2_score(y, pred)
    r2_log = r2_score(np.log(y), np.log(np.clip(pred, 1e-4, None)))
    rho = stats.spearmanr(y, pred).statistic
    print(f"[{etiqueta}] R²: {r2:.3f}  ·  RMSE: {rmse:.3f}  ·  MAE: {mae:.3f}  ·  MedAE: {medae:.3f}  ·  "
          f"R² en escala log: {r2_log:.3f}  ·  Spearman: {rho:.3f}")
    return dict(r2=r2, rmse=rmse, mae=mae, medae=medae, r2_log=r2_log, spearman=rho)


def ver_brecha(nombre, entrenamiento, prueba, umbral=15):
    # CDM-7: brecha relativa entre el desempeño de entrenamiento y el de prueba (<= 15 %)
    brecha = 100 * (entrenamiento - prueba) / entrenamiento
    estado = "OK" if abs(brecha) <= umbral else f"por encima del {umbral} % de CDM-7"
    print(f"{nombre}: entrena {entrenamiento:.3f}  ·  prueba {prueba:.3f}  ·  brecha {brecha:.1f} % ({estado})")
    return brecha


def lecturas_prioridad(y, prob, ha, titulo, sufijo, etiqueta_modelo, color="#2a78d6"):
    # Las cuatro lecturas de la misma probabilidad que usa el gradient boosting (5.1 y 6.1): ROC,
    # precision-recall, ganancia (OBJ-N1) y calibracion. Devuelve la cobertura al revisar 10, 20 y 30 %.
    y, prob, ha = np.asarray(y), np.asarray(prob), np.asarray(ha, dtype=float)
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    fpr, tpr, _ = roc_curve(y, prob)
    auc = roc_auc_score(y, prob)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{etiqueta_modelo} (AUC = {auc:.3f})")
    axes[0].plot([0, 1], [0, 1], color="grey", ls="--", lw=1, label="Al azar")
    axes[0].set(xlabel="Tasa de falsos positivos", ylabel="Tasa de verdaderos positivos", title=f"Curva ROC\n{titulo}")
    axes[0].legend(loc="lower right", fontsize=9)

    prec_c, rec_c, _ = precision_recall_curve(y, prob)
    ap = average_precision_score(y, prob)
    axes[1].plot(rec_c, prec_c, color=color, lw=2, label=f"{etiqueta_modelo} (AP = {ap:.3f})")
    axes[1].axhline(y.mean(), color="grey", ls="--", lw=1, label=f"Al azar (= {y.mean():.2f})")
    axes[1].set(xlabel="Recall (sensibilidad)", ylabel="Precisión", title=f"Curva precisión-recall\n{titulo}")
    axes[1].legend(loc="lower left", fontsize=9)

    orden = np.argsort(-prob)
    y_ord, ha_ord = y[orden], ha[orden]
    cubierto = np.cumsum(y_ord) / y_ord.sum()
    ha_cubierta = np.cumsum(ha_ord) / ha_ord.sum()
    fraccion = np.arange(1, len(y_ord) + 1) / len(y_ord)
    axes[2].plot(100 * fraccion, 100 * cubierto, color=color, lw=2, label="% de municipios-año que sí deforestaron")
    axes[2].plot(100 * fraccion, 100 * ha_cubierta, color="#2e9e5b", lw=2, ls="-.", label="% de hectáreas realmente deforestadas")
    axes[2].plot([0, 100], [0, 100], color="grey", ls="--", lw=1, label="Priorizar al azar")
    resultado = {}
    for corte in (0.1, 0.2, 0.3):
        n = int(corte * len(y_ord))
        resultado[int(100 * corte)] = dict(n=n, pct_municipios=100 * cubierto[n - 1], pct_ha=100 * ha_cubierta[n - 1],
                                           ha=ha_ord[:n].sum(), ha_total=ha_ord.sum())
        axes[2].annotate(f"{100 * cubierto[n - 1]:.0f} %", (100 * corte, 100 * cubierto[n - 1]),
                          textcoords="offset points", xytext=(5, -10), fontsize=9)
    axes[2].set(xlabel="% de municipios-año revisados", ylabel="% cubierto", title=f"Curva de ganancia (OBJ-N1)\n{titulo}")
    axes[2].legend(loc="lower right", fontsize=8)

    frac_pos, prob_media = calibration_curve(y, prob, n_bins=8, strategy="quantile")
    axes[3].plot(prob_media, frac_pos, "o-", color=color, lw=2, label=etiqueta_modelo)
    axes[3].plot([0, 1], [0, 1], color="grey", ls="--", lw=1, label="Calibración perfecta")
    axes[3].set(xlabel="Probabilidad predicha (promedio del grupo)", ylabel="Fracción que sí deforestó",
                title=f"Curva de calibración\n{titulo}")
    axes[3].legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.show()
    for corte, r in resultado.items():
        print(f"Revisando el {corte} % con mayor riesgo predicho ({r['n']} municipios-año, {sufijo}) se cubre el "
              f"{r['pct_municipios']:.0f} % de los que sí deforestaron y el {r['pct_ha']:.0f} % de las hectáreas "
              f"realmente deforestadas ({r['ha']:,.0f} de {r['ha_total']:,.0f} ha).")
    return resultado


def matriz_confusion(y, pred, titulo):
    matriz = confusion_matrix(y, pred)
    fig, ax = plt.subplots(figsize=(5, 4.5))
    sns.heatmap(matriz, annot=True, fmt=",d", cmap="Blues", cbar=False,
                xticklabels=["No deforesta", "Deforesta"], yticklabels=["No deforesta", "Deforesta"], ax=ax)
    ax.set(xlabel="Predicho", ylabel="Real", title=titulo)
    plt.show()
    return matriz

In [ ]:
clasificador = modelo_clas(params_clas).fit(X_train, y_train_clas)

pred_train = clasificador.predict(X_train)
pred_test = clasificador.predict(X_test)
prob_test = clasificador.predict_proba(X_test)[:, 1]

ba_train, ba_test = balanced_accuracy_score(y_train_clas, pred_train), balanced_accuracy_score(y_test_clas, pred_test)
ba_cv_dep = mejor_fila.metrica
brecha_clas = ver_brecha(f"Exactitud balanceada — entrenamiento vs. prueba {ANIO_PRUEBA}", ba_train, ba_test)
brecha_clas_cv = ver_brecha(f"Exactitud balanceada — validación por departamentos ({ANIO_INICIO}-{ANIO_PRUEBA - 1}) vs. prueba {ANIO_PRUEBA}", ba_cv_dep, ba_test)
auc_test = roc_auc_score(y_test_clas, prob_test)
print(f"AUC ({ANIO_PRUEBA}): {auc_test:.3f}")
m_clas_test = metricas_clasificacion(y_test_clas, pred_test, prob_test, f"prueba {ANIO_PRUEBA}")
matriz = matriz_confusion(y_test_clas, pred_test, f"Matriz de confusión — {ANIO_PRUEBA}")

### 5.1 Curva ROC, precisión-recall, ganancia y calibración (para priorizar, no solo para clasificar)

Las mismas cuatro lecturas de la misma probabilidad predicha que en el gradient boosting: **ROC** (minería de datos),
**precisión-recall** (un Observatorio con recursos limitados de verificación), **ganancia** (¿cuánto cubro si reviso solo
el 20 %?, OBJ-N1) y **calibración** (¿puedo usar la probabilidad como número?). Si la validación eligió `balanced`, las
probabilidades están desplazadas hacia la clase minoritaria y la curva de calibración se despega de la diagonal aunque el orden
(ROC, ganancia) sea igual de bueno.

In [ ]:
cobertura_2025 = lecturas_prioridad(y_test_clas, prob_test, test.def_ideam_ha.fillna(0), f"({ANIO_PRUEBA})",
                                    f"prueba {ANIO_PRUEBA}", "Regresión logística", color="#7b4fb3")

**Lectura: qué significa este desempeño para OBJ-N1 y CDM-7.**

- **Sirve para territorios que no vio.** El AUC es 0,906 en la prueba temporal 2025 y 0,905 con departamentos completos fuera; la exactitud balanceada, 0,826 y 0,825. No hay pérdida al
  pasar de "mismo territorio, año nuevo" a "departamento nuevo", que es la condición real del Observatorio (priorizar donde todavía no ha mirado).
- **La brecha entrenamiento-prueba es de 0,7 %** (CDM-7 pide ≤ 15 %): un modelo lineal regularizado casi no memoriza.
- **Cómo leer la curva de ganancia.** Revisando el 10 %, 20 % y 30 % de los municipios-año con mayor riesgo se cubre el 15 %, 31 % y 46 % de los que deforestan, pero el 78 %, 89 % y 96 % de las
  **hectáreas**. La cobertura de municipios parece baja, pero casi llega al tope matemático: como el 64 % de los municipios-año deforesta (410 de 639), revisar 63 municipios solo puede cubrir 15,4 %
  (63/410), 127 puede cubrir 31,0 % y 191, 46,6 %. Es decir, los municipios que el modelo pone arriba casi siempre sí deforestan; lo que no se puede es "descartar" fácilmente a los demás, porque la mayoría
  deforesta algo. Por eso el argumento para el Observatorio es el de las hectáreas: con el 30 % de la lista se cubre casi toda la deforestación.
- **Al umbral 0,5**, de cada 10 municipios que marca, 9 sí deforestan (precisión 0,91) y deja pasar 2 de cada 10 que deforestan (recall 0,80); el costo de las falsas alarmas es una visita de verificación
  que no encuentra deforestación (especificidad 0,85).

In [ ]:
en_deforesta_test = (y_test_clas == 1).values
regresor = modelo_reg(params_reg).fit(X_train[en_deforesta], y_train_tasa[en_deforesta])

pred_train_tasa = regresor.predict(X_train[en_deforesta])
pred_test_tasa = regresor.predict(X_test[en_deforesta_test])
y_test_tasa_real = y_test_tasa[en_deforesta_test]

r2_train = r2_score(y_train_tasa[en_deforesta], pred_train_tasa)
r2_test_ = r2_score(y_test_tasa_real, pred_test_tasa)
brecha_reg = ver_brecha(f"R² — entrenamiento vs. prueba {ANIO_PRUEBA} (solo deforesta > 0)", r2_train, r2_test_) if r2_train > 0 else np.nan
m_reg_test = metricas_regresion(y_test_tasa_real, pred_test_tasa, f"prueba {ANIO_PRUEBA}")
r2_test, mae_test, medae_test, rmse_test = m_reg_test["r2"], m_reg_test["mae"], m_reg_test["medae"], m_reg_test["rmse"]
print("(MedAE y Spearman son más robustos que MAE/RMSE/R² frente a los pocos casos extremos de 5.3)")

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].scatter(y_test_tasa_real, pred_test_tasa, alpha=.5, color="#7b4fb3")
lim = max(y_test_tasa_real.max(), pred_test_tasa.max())
ax[0].plot([0, lim], [0, lim], color="grey", ls="--", lw=1)
ax[0].set(xlabel="Tasa real (%)", ylabel="Tasa predicha (%)", title=f"Dado que deforesta, ¿cuánto? — {ANIO_PRUEBA} (escala natural)")
ax[1].scatter(y_test_tasa_real, np.clip(pred_test_tasa, 1e-4, None), alpha=.5, color="#7b4fb3")
ax[1].plot([1e-3, lim], [1e-3, lim], color="grey", ls="--", lw=1)
ax[1].set(xscale="log", yscale="log", xlabel="Tasa real (%, escala log)", ylabel="Tasa predicha (%, escala log)",
          title="Lo mismo en escala logarítmica")
plt.tight_layout()
plt.show()

### 5.2 Evidencia: por qué separar en clasificación + regresión condicional

El mismo experimento del gradient boosting. Se entrena, solo para esta comparación, un modelo de **una sola parte** con
todas las filas de 2021-2024 (ceros incluidos). Como una Gamma no admite ceros, la versión de una parte usa una
regresión **Tweedie** (potencia 1,5, enlace logarítmico), la extensión estándar de la Gamma para datos con ceros, con
el mismo `alpha` ya elegido. Se compara **donde de verdad importa la magnitud**: las filas de 2025 que sí deforestaron.

In [ ]:
directo = pipe_lineal(TweedieRegressor(power=1.5, link="log", max_iter=5000, **params_reg), VARS_LR).fit(X_train, y_train_tasa)
pred_directo_def = directo.predict(X_test)[en_deforesta_test]

comparacion = pd.DataFrame([
    {"enfoque": "Una sola regresión (Tweedie, entrenada con TODAS las filas, ceros incluidos)",
     "R²": r2_score(y_test_tasa_real, pred_directo_def), "MAE": mean_absolute_error(y_test_tasa_real, pred_directo_def),
     "predicción media": pred_directo_def.mean()},
    {"enfoque": "Regresión condicional (Gamma, entrenada solo con deforesta == 1, este notebook)",
     "R²": r2_test, "MAE": mae_test, "predicción media": pred_test_tasa.mean()},
])
print(f"Evaluado solo sobre las {en_deforesta_test.sum()} filas de {ANIO_PRUEBA} que SÍ deforestaron "
      f"(tasa real promedio: {y_test_tasa_real.mean():.3f}):")
display(comparacion.round(3))
print(f"La predicción media sobre los municipios que sí deforestaron es {comparacion.loc[0, 'predicción media']:.3f} con una sola "
      f"regresión y {comparacion.loc[1, 'predicción media']:.3f} con la condicional (la real es {y_test_tasa_real.mean():.3f}).")
if comparacion.loc[1, "predicción media"] > comparacion.loc[0, "predicción media"]:
    print("Los ceros estructurales jalan la predicción hacia abajo en la regresión de una sola parte; la condicional queda más cerca de la real.")
else:
    print("Aquí la separación no acerca la predicción media a la real: conviene decirlo tal cual en el documento.")

### 5.3 Por qué el R² de "¿cuánto?" es tan sensible a un solo municipio (y qué lecturas no lo son)

El R² es un promedio de errores al cuadrado: una sola fila con un error enorme pesa como decenas con error normal. **El
caso es Pedraza (Magdalena)**: su bosque IDEAM es de apenas ~9 ha, y como la tasa se calcula sobre el bosque que queda, perder 8 ha
se vuelve una tasa de 88 %, muy por fuera de cualquier otro municipio rural. Ninguna de las 15 variables distingue "casi no
queda bosque". Como en el gradient boosting, el R² reportado no recorta ni borra la fila; el cálculo sin Pedraza es un
diagnóstico de sensibilidad, y se miran dos lecturas que no dependen de ese extremo: el R² en escala logarítmica y la
correlación de Spearman.

In [ ]:
test_def = test[en_deforesta_test]
es_pedraza = (test_def.cod_dane == "47541").values

if es_pedraza.any():
    print(f"Tasa real de Pedraza ({ANIO_PRUEBA}): {y_test_tasa_real.values[es_pedraza][0]:.1f} %  ·  "
          f"predicha por el modelo: {pred_test_tasa[es_pedraza][0]:.1f} %")
    sin_pedraza = ~es_pedraza
    r2_sin_pedraza = r2_score(y_test_tasa_real.values[sin_pedraza], pred_test_tasa[sin_pedraza])
    mae_sin_pedraza = mean_absolute_error(y_test_tasa_real.values[sin_pedraza], pred_test_tasa[sin_pedraza])
    print()
    brecha_sin_pedraza = ver_brecha("R² excluyendo solo a Pedraza (diagnóstico)", r2_train, r2_sin_pedraza) if r2_train > 0 else np.nan
    print(f"Error absoluto promedio excluyendo a Pedraza: {mae_sin_pedraza:.3f} (vs. {mae_test:.3f} con Pedraza incluido)")
    print(f"\nLecturas robustas, con Pedraza incluido: R² en escala log = {m_reg_test['r2_log']:.3f}  ·  "
          f"Spearman = {m_reg_test['spearman']:.3f}")
else:
    print("Pedraza no está entre los municipios que deforestaron en el año de prueba: se omite este diagnóstico.")
    r2_sin_pedraza = mae_sin_pedraza = brecha_sin_pedraza = np.nan

**Lectura: qué sí y qué no se puede decir de "¿cuánto?" (CDM-7).**

El R² de prueba es 0,013 y la brecha contra el entrenamiento es 89 %: **no cumple CDM-7**, y la causa no es sobreajuste sino un solo municipio. Sin Pedraza, el R² de prueba es 0,112 y la brecha 4 %, que sí cumple.
Pero 0,11 sigue siendo poco: con estas 15 variables la magnitud de la deforestación no se puede **cuantificar** (no sirve para estimar hectáreas). Lo que sí soporta es **ordenar** municipios por severidad esperada
(correlación de Spearman 0,49), y leer asociaciones (7.2). Es una limitación que conviene declarar tal cual en el documento, en lugar de forzar un R² más alto: los saltos grandes
y poco frecuentes del corredor seco del Caribe no tienen ninguna variable que los distinga.

### 5.4 ¿Qué aportan el crédito y la producción más allá de lo estructural?

El gradient boosting reporta el modelo completo; no dice cuánto del desempeño se debe a lo que ya sabe cualquiera (el
municipio tiene bosque y es grande) y cuánto a las variables financieras y productivas que son el motivo del proyecto.
Con los mismos hiperparámetros y evaluando en 2025:

| Especificación | Variables |
|---|---|
| **Persistencia** (línea base, sin modelo) | La frecuencia con que deforestaron en el entrenamiento los municipios que sí / no deforestaron el año anterior |
| **A. Estructural** | `bosque`, `area` |
| **B. Principal** | Las variables que sobreviven al VIF (el modelo de este notebook) |
| **B15. Sin podar** | Las 15 variables, sin eliminar ninguna por VIF (regularizadas): ¿cuesta algo podar? |
| **C. Con persistencia** | B + si el municipio deforestó el año anterior |
| **D. Con región** | B + la región (categórica) |

C necesita la deforestación del año anterior, que no existe para 2021, así que **todas las especificaciones de esta
sección se entrenan con 2022-2024**.

In [ ]:
panel["deforesta_l1"] = panel.groupby("cod_dane").tasa.transform(
    lambda s: (s.shift(1) > 0).astype(float).where(s.shift(1).notna()))
extra = (panel[["cod_dane", "anio", "deforesta_l1"]]
         .merge(anual[["cod_dane", "anio", "region"]], on=["cod_dane", "anio"], how="left"))
train_e = train.merge(extra, on=["cod_dane", "anio"], how="left")
test_e = test.merge(extra, on=["cod_dane", "anio"], how="left")
train_c = train_e[train_e.anio >= ANIO_INICIO + 1].reset_index(drop=True)


def ajustar_spec(cols_log, cols_extra=(), cols_cat=()):
    pasos = [("log", Pipeline([("l", FunctionTransformer(np.log1p)), ("e", StandardScaler())]), list(cols_log))]
    if cols_extra:
        pasos.append(("ext", StandardScaler(), list(cols_extra)))
    if cols_cat:
        pasos.append(("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), list(cols_cat)))
    modelo = Pipeline([("pre", ColumnTransformer(pasos)),
                       ("m", LogisticRegression(solver="liblinear", max_iter=5000, random_state=SEMILLA, **params_clas))])
    return modelo.fit(train_c[list(cols_log) + list(cols_extra) + list(cols_cat)], train_c.deforesta)


especificaciones = {"A. Estructural": (["bosque", "area"], (), ()),
                    "B. Principal (tras VIF)": (VARS_LR, (), ()),
                    "B15. Sin podar (15 variables)": (EXPLICATIVAS, (), ()),
                    "C. Con persistencia": (VARS_LR, ["deforesta_l1"], ()),
                    "D. Con región": (VARS_LR, (), ["region"])}
tasa_cond = train_c.groupby("deforesta_l1").deforesta.mean()
prob_pers = test_e.deforesta_l1.map(tasa_cond).values
filas = [{"especificación": "Persistencia (año anterior, sin modelo)", "AUC": roc_auc_score(y_test_clas, prob_pers),
          "exactitud balanceada": balanced_accuracy_score(y_test_clas, prob_pers >= UMBRAL),
          "AP": average_precision_score(y_test_clas, prob_pers)}]
for nombre, (lg, ex, ca) in especificaciones.items():
    mod = ajustar_spec(lg, ex, ca)
    prob = mod.predict_proba(test_e[list(lg) + list(ex) + list(ca)])[:, 1]
    filas.append({"especificación": nombre, "AUC": roc_auc_score(y_test_clas, prob),
                  "exactitud balanceada": balanced_accuracy_score(y_test_clas, prob >= UMBRAL),
                  "AP": average_precision_score(y_test_clas, prob)})
especificaciones_tabla = pd.DataFrame(filas).set_index("especificación")
display(especificaciones_tabla.round(4))
e = especificaciones_tabla.AUC
print(f"El bosque y el área solos (A) logran un AUC de {e['A. Estructural']:.3f}; con el modelo principal (B) sube a "
      f"{e['B. Principal (tras VIF)']:.3f}: el crédito, los cultivos y el ganado aportan {e['B. Principal (tras VIF)'] - e['A. Estructural']:+.3f} de AUC.")
print(f"Podar por VIF cambia el AUC en {e['B. Principal (tras VIF)'] - e['B15. Sin podar (15 variables)']:+.3f} frente a usar las 15 (positivo = podar no pierde nada); "
      f"agregar persistencia cambia {e['C. Con persistencia'] - e['B. Principal (tras VIF)']:+.3f}; agregar región, {e['D. Con región'] - e['B. Principal (tras VIF)']:+.3f}.")

**Lectura: qué aportan el crédito y la producción, y qué no (ODM-3, OBJ-N1).**

- Solo con bosque y área (A) el AUC es 0,896; con el modelo principal (B), 0,904: **crédito, cultivos y ganado agregan +0,008 de AUC** sobre lo estructural. Saber si el municipio deforestó el año anterior (C) agrega más (+0,016),
  y la persistencia sola, sin modelo, ya da 0,827. Podar por VIF no pierde nada (+0,002) y la región no agrega nada.
- **Qué significa para el objetivo.** La capacidad de priorizar viene sobre todo de **dónde hay bosque y de qué tamaño es el municipio** (la variable objetivo "deforesta > 0" favorece mecánicamente a los municipios con mucho bosque).
  Las variables financieras y productivas no son el motor de la predicción. Por eso el Observatorio no debe presentar el modelo como "el crédito predice la deforestación": su aporte es (1) una priorización territorial que
  funciona en departamentos nuevos y (2) una lectura de **qué asociaciones financiero-productivas existen, en qué dirección y con cuánta incertidumbre** (ODM-3), que es justo lo que pide el objetivo y no una mejora de AUC.
- **Una limitación real.** Que el aporte predictivo del crédito sea pequeño no prueba que el crédito sea irrelevante: las variables son del mismo año, están correlacionadas entre sí y el crédito se mide por km², no por
  destino de cada préstamo. Es un hallazgo sobre estos datos, no sobre el mundo.

## 6. Evaluación dejando departamentos completos fuera

La sección 5 mide generalización en el tiempo, pero los 639 municipios de 2025 son los mismos que el modelo vio entrenando.
Falta medir generalización en el **territorio**: `GroupKFold` por departamento (5 pliegues) sobre **todo 2021-2025**, con los
hiperparámetros ya elegidos (sin volver a buscar ni a calcular el VIF). Cada fila queda predicha por un modelo que nunca vio
su departamento: es la prueba de que sirve para priorizar territorios nuevos (OBJ-N1).

In [ ]:
m = muestra.reset_index(drop=True)
grupos_todos = m.nombre_departamento
gkf_eval = GroupKFold(n_splits=5)

pred_clas_oof = pd.Series(np.nan, index=m.index)
prob_clas_oof = pd.Series(np.nan, index=m.index)
pred_tasa_oof = pd.Series(np.nan, index=m.index)

for i_dentro, i_fuera in gkf_eval.split(m[EXPLICATIVAS], m.deforesta, groups=grupos_todos):
    clas_fold = modelo_clas(params_clas).fit(m[EXPLICATIVAS].iloc[i_dentro], m.deforesta.iloc[i_dentro])
    pred_clas_oof.iloc[i_fuera] = clas_fold.predict(m[EXPLICATIVAS].iloc[i_fuera])
    prob_clas_oof.iloc[i_fuera] = clas_fold.predict_proba(m[EXPLICATIVAS].iloc[i_fuera])[:, 1]

    en_def_dentro = (m.deforesta.iloc[i_dentro] == 1).values
    reg_fold = modelo_reg(params_reg).fit(m[EXPLICATIVAS].iloc[i_dentro][en_def_dentro], m.tasa.iloc[i_dentro][en_def_dentro])
    pred_tasa_oof.iloc[i_fuera] = reg_fold.predict(m[EXPLICATIVAS].iloc[i_fuera])

en_def_oof = (m.deforesta == 1).values
ba_departamentos = balanced_accuracy_score(m.deforesta, pred_clas_oof)

print(f"Departamentos en la muestra: {grupos_todos.nunique()}")
print(f"Exactitud balanceada, ¿deforesta? (departamentos fuera): {ba_departamentos:.3f}  "
      f"(referencia, prueba temporal {ANIO_PRUEBA}: {ba_test:.3f})")
m_clas_oof = metricas_clasificacion(m.deforesta, pred_clas_oof, prob_clas_oof, "departamentos fuera")

m_reg_oof = metricas_regresion(m.tasa[en_def_oof], pred_tasa_oof[en_def_oof], "departamentos fuera")
r2_departamentos, mae_departamentos = m_reg_oof["r2"], m_reg_oof["mae"]
medae_departamentos, rmse_departamentos = m_reg_oof["medae"], m_reg_oof["rmse"]
print(f"(referencia, prueba temporal {ANIO_PRUEBA} — R²: {r2_test:.3f}  ·  MAE: {mae_test:.3f}  ·  "
      f"MedAE: {medae_test:.3f}  ·  RMSE: {rmse_test:.3f})")

### 6.1 Matriz de confusión, ROC, precisión-recall, ganancia y calibración (departamentos fuera)

Las mismas lecturas de 5/5.1 con las predicciones "limpias" de la sección 6: la versión más exigente, la que responde si el modelo
sirve para un departamento que el Observatorio todavía no ha mirado (OBJ-N1).

In [ ]:
matriz_oof = matriz_confusion(m.deforesta, pred_clas_oof, "Matriz de confusión — departamentos fuera")
cobertura_oof = lecturas_prioridad(m.deforesta, prob_clas_oof, m.def_ideam_ha.fillna(0), "(departamentos fuera)",
                                   "sin que el modelo haya visto su departamento", "Regresión logística", color="#7b4fb3")

### 6.2 Los errores más grandes de "¿cuánto?" (departamentos fuera)

Igual que en 5.3: qué filas concretas producen el R². Importa ver si los errores más grandes son casos de bosque casi nulo o saltos
reales poco frecuentes que ningún modelo con estas variables podría anticipar.

In [ ]:
errores = pd.DataFrame({
    "cod_dane": m.cod_dane[en_def_oof], "municipio": m.nombre_municipio[en_def_oof], "departamento": m.nombre_departamento[en_def_oof],
    "anio": m.anio[en_def_oof], "tasa_real": m.tasa[en_def_oof], "tasa_predicha": pred_tasa_oof[en_def_oof],
})
errores["error_absoluto"] = (errores.tasa_real - errores.tasa_predicha).abs()
errores = errores.sort_values("error_absoluto", ascending=False)
print("Las 6 filas con mayor error absoluto (departamentos fuera):")
errores.head(6).round(2)

### 6.3 Riesgo predicho por departamento (lectura de negocio)

Las probabilidades de la sección 6 ya son "limpias" y se pueden agregar como priorización territorial (OBJ-N1). Se muestran los diez
departamentos con mayor probabilidad promedio, junto con la frecuencia real de municipios-año que deforestaron.

In [ ]:
por_depto = pd.DataFrame({
    "riesgo_predicho": prob_clas_oof.groupby(m.nombre_departamento).mean(),
    "tasa_deforesta_real": m.groupby("nombre_departamento").deforesta.mean(),
    "municipios_anio": m.groupby("nombre_departamento").size(),
}).sort_values("riesgo_predicho", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5.5))
y_pos = np.arange(len(por_depto))
ax.barh(y_pos - 0.2, por_depto.riesgo_predicho, height=0.4, color="#7b4fb3", label="Riesgo predicho (modelo)")
ax.barh(y_pos + 0.2, por_depto.tasa_deforesta_real, height=0.4, color="#2a78d6", label="% de municipios-año que sí deforestaron")
ax.set_yticks(y_pos, por_depto.index)
ax.invert_yaxis()
ax.set(xlabel="Proporción", title="Los 10 departamentos de mayor riesgo predicho (OBJ-N1, con departamentos fuera)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=10, frameon=False)
plt.tight_layout()
plt.show()
por_depto.round(3)

## 7. Importancia e inferencia (ODM-3)

Qué líneas de crédito y qué cadenas se asocian con la deforestación, en qué dirección y con cuánta certeza: la tercera pregunta
específica de negocio, y lo que la logística hace mejor que cualquier ensamble.

Igual que en el gradient boosting, **no todas las variables son del mismo tipo**: `bosque` y `area` son **estructurales** (nadie las
"mueve" con una política) y el resto son **variables de intervención** (crédito, cultivos, ganado). Por eso la comparación se
separa en dos preguntas (7.1).

**Tres lecturas de la misma verdad, de menor a mayor exigencia.** (1) El **peso estandarizado** de cada coeficiente en el
modelo regularizado: el análogo de la "ganancia" del gradient boosting. (2) La **inferencia clásica** con
errores robustos por conglomerado (7.2): *odds ratios* con intervalo, valor p, corrección de Holm, efectos marginales y pruebas
conjuntas por actividad; es la que pide CDM-5. (3) La **permutación** (7.5), idéntica a la de los otros dos modelos, que mide qué
tanto necesita el modelo cada variable para predecir en 2025.

In [ ]:
ESTRUCTURALES = {"bosque", "area"}
POLITICA = [v for v in EXPLICATIVAS if v not in ESTRUCTURALES]
CREDITO = ["cred", "cred_gan", "cred_palma", "cred_cacao", "cred_cafe", "cred_arroz"]
nombres_politica = [NOMBRES[v] for v in POLITICA]
nombres_credito = [NOMBRES[v] for v in CREDITO]

# Bloques y actividades: los mismos en los tres modelos. Se barajan JUNTAS todas las columnas del grupo,
# para no subestimar variables que se reparten la importancia porque estan correlacionadas entre si
# (credito y area de cafe, credito ganadero y bovinos/fincas...).
BLOQUES = {"Crédito (6 líneas)": CREDITO,
           "Cultivos (5 columnas de % del área)": ["agricola", "palma", "cacao", "cafe", "arroz"],
           "Ganado (bovinos y fincas)": ["bovinos", "fincas"],
           "Estructura (bosque y área)": ["bosque", "area"]}
ACTIVIDADES = {"Ganadería (crédito + bovinos + fincas)": ["cred_gan", "bovinos", "fincas"],
               "Café (crédito + área)": ["cred_cafe", "cafe"],
               "Cacao (crédito + área)": ["cred_cacao", "cacao"],
               "Palma (crédito + área)": ["cred_palma", "palma"],
               "Arroz (crédito + área)": ["cred_arroz", "arroz"],
               "Escala total (crédito y área cosechada totales)": ["cred", "agricola"]}


def auc_rapido(y, p):
    # AUC por rangos (Mann-Whitney); equivale a roc_auc_score pero es ~10 veces mas rapido, y se usa miles de veces
    r = stats.rankdata(p)
    n1 = y.sum()
    n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def permutacion_con_p(modelo, X, y, tipo, grupos, usadas=None, n_repeticiones=30, n_perm_boot=10, n_boot=400):
    # Importancia por permutacion sobre 2025, con DOS valores p:
    #  - p_valor: el del gradient boosting (prueba t de las 30 repeticiones contra cero). Solo recoge el azar del
    #    barajado, no la incertidumbre de tener 639 municipios: es optimista, y se deja para poder comparar.
    #  - p_territorio: bootstrap por DEPARTAMENTO (se remuestrean departamentos completos, como en los errores
    #    robustos por conglomerado espacial que pide CDM-5). Es la prueba que se usa para contar variables.
    # p_holm corrige p_territorio por comparaciones multiples (Holm) entre las variables que el modelo usa.
    scoring, predecir, metrica = (("roc_auc", lambda mod, Z: mod.predict_proba(Z)[:, 1], auc_rapido) if tipo == "clas"
                                  else ("r2", lambda mod, Z: mod.predict(Z), r2_score))
    y = np.asarray(y)
    resultado = permutation_importance(modelo, X, y, n_repeats=n_repeticiones, random_state=SEMILLA,
                                        scoring=scoring, n_jobs=1)
    rng = np.random.default_rng(SEMILLA)
    p0 = predecir(modelo, X)
    perms = {}
    for col in X.columns:
        ps = []
        for _ in range(n_perm_boot):
            Xp = X.copy()
            Xp[col] = X[col].values[rng.permutation(len(X))]
            ps.append(predecir(modelo, Xp))
        perms[col] = np.array(ps)
    g = np.asarray(grupos)
    deptos = np.unique(g)
    idx_depto = {d: np.where(g == d)[0] for d in deptos}
    caidas = {col: [] for col in X.columns}
    for _ in range(n_boot):
        idx = np.concatenate([idx_depto[d] for d in rng.choice(deptos, size=len(deptos), replace=True)])
        base = metrica(y[idx], p0[idx])
        for col in X.columns:
            caidas[col].append(base - np.mean([metrica(y[idx], pp[idx]) for pp in perms[col]]))
    filas = []
    for i, var in enumerate(X.columns):
        est = metrica(y, p0) - np.mean([metrica(y, pp) for pp in perms[var]])
        se = np.std(caidas[var], ddof=1)
        t_p = stats.ttest_1samp(resultado.importances[i], 0).pvalue
        filas.append({"variable": NOMBRES[var], "clave": var, "importancia_media": resultado.importances_mean[i],
                      "desv": resultado.importances_std[i], "p_valor": 1.0 if np.isnan(t_p) else t_p,
                      "ic95_inf_territorio": est - 1.96 * se, "ic95_sup_territorio": est + 1.96 * se,
                      "p_territorio": 1.0 if (se == 0 or np.isnan(se)) else stats.norm.sf(est / se)})
    t = pd.DataFrame(filas).sort_values("importancia_media", ascending=False).reset_index(drop=True)
    usadas = list(X.columns) if usadas is None else list(usadas)
    t["p_holm"] = np.nan
    en_uso = t.clave.isin(usadas)
    t.loc[en_uso, "p_holm"] = multipletests(t.loc[en_uso, "p_territorio"], method="holm")[1]
    t["tipo"] = np.where(t.clave.isin(ESTRUCTURALES), "estructural", "intervención")
    return t


def permutar_bloques(modelo, X, y, grupos, puntaje, n_repeats=30, semilla=SEMILLA):
    # Caida de la metrica al desordenar juntas (con la misma permutacion de filas) las columnas de cada grupo.
    rng = np.random.default_rng(semilla)
    base = puntaje(modelo, X, y)
    filas = []
    for nombre, cols in grupos.items():
        caidas = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[cols] = X[cols].values[rng.permutation(len(X))]
            caidas.append(base - puntaje(modelo, Xp, y))
        filas.append({"grupo": nombre, "n_variables": len(cols), "caida": np.mean(caidas), "desv": np.std(caidas),
                      "p_valor": stats.ttest_1samp(caidas, 0).pvalue})
    return pd.DataFrame(filas).set_index("grupo").sort_values("caida", ascending=False)


def puntaje_auc(modelo, X, y):
    return roc_auc_score(y, modelo.predict_proba(X)[:, 1])


def puntaje_r2(modelo, X, y):
    return r2_score(y, modelo.predict(X))


def graficar_bloques(tablas, titulo_metrica):
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
    for eje, (clave, titulo) in zip(ax, [("bloques", "Por bloque temático"), ("actividades", "Por actividad productiva")]):
        t = tablas[clave].sort_values("caida")
        eje.barh(t.index, t.caida, xerr=t.desv, color="#2a78d6", capsize=3)
        eje.axvline(0, ls="--", color="grey")
        eje.set_xlabel(f"caída de {titulo_metrica} al barajar el grupo ({ANIO_PRUEBA})")
        eje.set_title(titulo)
    plt.tight_layout()
    plt.show()


from matplotlib.patches import Patch

pre_f = clasificador.named_steps["pre"]
lg = clasificador.named_steps["m"]
Z_train = pd.DataFrame(pre_f.transform(X_train), columns=VARS_LR, index=X_train.index)
sd_log = pd.Series(pre_f.named_transformers_["num"].named_steps["esc"].scale_, index=VARS_LR)

pesos = pd.DataFrame({"peso": lg.coef_.ravel()}, index=VARS_LR)
pesos["OR_por_sd"] = np.exp(pesos.peso)
pesos["OR_duplicar"] = np.exp(pesos.peso / sd_log * np.log(2))  # aproximado: vale para valores mayores que 1
pesos["peso_estandarizado"] = pesos.peso.abs()                    # las variables ya estan en desviaciones estandar
pesos["participacion_%"] = 100 * pesos.peso_estandarizado / pesos.peso_estandarizado.sum()
pesos["direccion"] = np.where(pesos.peso > 0, "sube", np.where(pesos.peso < 0, "baja", "sin efecto"))
pesos["tipo"] = np.where(pesos.index.isin(ESTRUCTURALES), "estructural", "intervención")
pesos = pesos.sort_values("peso_estandarizado", ascending=False)
cero = pesos.index[pesos.peso == 0].map(NOMBRES).tolist()
print(f"Intercepto: {lg.intercept_[0]:.3f}  ·  coeficientes llevados a cero por la penalización: {cero or 'ninguno'}")

d = pesos.sort_values("peso")
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh([NOMBRES[v] for v in d.index], d.peso, color=["#9aa0a6" if t == "estructural" else ("#c8541a" if p > 0 else "#2a78d6")
                                                       for t, p in zip(d.tipo, d.peso)])
ax.axvline(0, color="black", lw=0.8)
ax.set(xlabel="peso β del modelo final (por +1 desv. estándar de log(1+x))", title="Pesos de la regresión logística — ¿deforesta?")
ax.legend(handles=[Patch(color="#9aa0a6", label="Estructural"), Patch(color="#c8541a", label="Sube la probabilidad"),
                    Patch(color="#2a78d6", label="Baja la probabilidad")], loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()
pesos.assign(variable=[NOMBRES[v] for v in pesos.index]).set_index("variable")[["tipo", "peso", "OR_por_sd", "OR_duplicar", "participacion_%", "direccion"]].round(3)

### 7.1 Dos preguntas distintas, dos lecturas distintas

**Pregunta 1 — ¿qué tan arriesgado es el municipio, en general?** Aquí `bosque` y `area` mandan, y no es un problema: son la variable
de **exposición**. Confirma que el modelo prioriza con el criterio correcto (OBJ-N1).

**Pregunta 2 — entre las variables que el Observatorio sí puede incidir, ¿cuál pesa más?** La que responde ODM-3. Se compara el
peso de las variables de intervención **entre sí**, y luego solo el de las líneas de crédito que el VIF dejó en el modelo.
Las eliminadas por VIF (ver 2.2) no aparecen: su información la lleva la variable que las representa.

In [ ]:
interv = pesos[pesos.tipo == "intervención"]
print("Participación (%) en el peso total, por tipo de variable:")
print(pesos.groupby("tipo")["participacion_%"].sum().round(1).to_string())
print("\nRanking entre las variables de intervención que quedan en el modelo (peso estandarizado):")
for v, r in interv.iterrows():
    print(f"  {NOMBRES[v]:42s} {r.peso:+.3f}  ({r.direccion}, OR por +1 desv. = {r.OR_por_sd:.2f})")
cred_modelo = [v for v in interv.index if v in CREDITO]
print("\nSOLO entre las líneas de crédito que quedan en el modelo:")
for v in cred_modelo:
    r = pesos.loc[v]
    print(f"  {NOMBRES[v]:42s} {r.peso:+.3f}  ({r.direccion})")
top_credito_lr = pesos.loc[cred_modelo].peso_estandarizado.idxmax()
print(f"\nRespuesta directa a ODM-3 (\"¿qué línea de crédito pesa más?\"): {NOMBRES[top_credito_lr]}, que "
      f"{'sube' if pesos.loc[top_credito_lr, 'peso'] > 0 else 'baja'} la probabilidad de deforestar.")

### 7.2 Inferencia: odds ratios, errores robustos por departamento y corrección de Holm (CDM-5)

Para inferir, el modelo se re-estima con `statsmodels`, **sin regularización ni pesos**, sobre las mismas variables
transformadas y estandarizadas. Los coeficientes son los de una logística clásica; lo que importa es su incertidumbre. Un mismo
municipio aparece cuatro años y municipios vecinos se parecen, así que las observaciones no son independientes. Se calculan los
errores estándar de dos formas:

- **Agrupados por departamento** (*cluster-robust*): permiten que todos los municipios de un departamento tengan errores
  correlacionados. Es la versión que pide CDM-5 ("conglomerado espacial") y la más conservadora (31 grupos).
- **Agrupados por municipio**: la versión del notebook original, como contraste.

El **odds ratio (OR)** se lee como: "+1 desviación estándar de la variable (en escala log) multiplica las odds de deforestar por
OR, manteniendo lo demás constante". Como CDM-5 exige corregir por comparaciones múltiples, se agrega el valor p de **Holm** sobre
los coeficientes de las variables del modelo. Y como CDM-5 habla de variables financieras y de producción, se cuentan **solo las de
intervención** (el bosque y el área siempre salen significativos y no cuentan).

In [ ]:
Xsm = sm.add_constant(Z_train)
cl_dep = pd.factorize(train.nombre_departamento)[0]
cl_mun = pd.factorize(train.cod_dane)[0]
logit_dep = sm.Logit(y_train_clas, Xsm).fit(disp=0, cov_type="cluster", cov_kwds={"groups": cl_dep})
logit_mun = sm.Logit(y_train_clas, Xsm).fit(disp=0, cov_type="cluster", cov_kwds={"groups": cl_mun})
print(logit_dep.summary2().tables[0].to_string(header=False, index=False))

coefs = pd.DataFrame({"coef": logit_dep.params, "OR": np.exp(logit_dep.params)}).drop("const")
for nombre, res in [("dep", logit_dep), ("mun", logit_mun)]:
    ic = res.conf_int().drop("const")
    coefs[f"OR_IC95_inf_{nombre}"] = np.exp(ic[0])
    coefs[f"OR_IC95_sup_{nombre}"] = np.exp(ic[1])
    coefs[f"p_{nombre}"] = res.pvalues.drop("const")
coefs["p_holm_dep"] = multipletests(coefs.p_dep, method="holm")[1]
coefs["tipo"] = np.where(coefs.index.isin(ESTRUCTURALES), "estructural", "intervención")
coefs["signif_holm"] = np.where(coefs.p_holm_dep < 0.05, "sí", "no")
coefs_tabla = coefs.assign(variable=[NOMBRES[v] for v in coefs.index]).set_index("variable").sort_values("OR", ascending=False)
print(f"\nPseudo-R² de McFadden: {logit_dep.prsquared:.3f}")
coefs_tabla[["tipo", "OR", "OR_IC95_inf_dep", "OR_IC95_sup_dep", "p_dep", "p_holm_dep", "p_mun", "signif_holm"]].round(4)

In [ ]:
d = coefs.sort_values("OR")
fig, ax = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
for eje, (suf, titulo) in zip(ax, [("dep", "Errores agrupados por departamento"), ("mun", "Errores agrupados por municipio")]):
    for signif, color in [(True, "#c8541a"), (False, "#9aa0a6")]:
        s = d[(d[f"p_{suf}"] < 0.05) == signif]
        eje.errorbar(s.OR, [NOMBRES[v] for v in s.index], xerr=[s.OR - s[f"OR_IC95_inf_{suf}"], s[f"OR_IC95_sup_{suf}"] - s.OR],
                     fmt="o", capsize=3, color=color)
    eje.axvline(1, ls="--", color="black"); eje.set_xscale("log")
    eje.set_xlabel("Odds ratio por +1 desv. estándar (escala log) e IC 95 %"); eje.set_title(titulo)
fig.suptitle("Asociación con tener alguna deforestación (tasa > 0) en el año — naranja: p < 0,05")
plt.tight_layout()
plt.show()

inter = coefs[coefs.tipo == "intervención"]
cdm5 = {"n_dep": int((inter.p_dep < 0.05).sum()), "n_holm": int((inter.p_holm_dep < 0.05).sum()), "n_mun": int((inter.p_mun < 0.05).sum()),
        "total": len(inter), "nombres": [NOMBRES[v] for v in inter.index[inter.p_holm_dep < 0.05]]}
print(f"CDM-5: de las {cdm5['total']} variables de intervención del modelo, {cdm5['n_mun']} tienen p < 0,05 con errores por municipio, "
      f"{cdm5['n_dep']} con errores por departamento y {cdm5['n_holm']} tras la corrección de Holm.")
for v in inter.index[inter.p_holm_dep < 0.05]:
    r = inter.loc[v]
    print(f"   {NOMBRES[v]:42s} OR = {r.OR:.2f}  (IC 95 % {r.OR_IC95_inf_dep:.2f}-{r.OR_IC95_sup_dep:.2f}; p = {r.p_dep:.3f}; Holm = {r.p_holm_dep:.3f})")
cambian = [NOMBRES[v] for v in inter.index if (inter.loc[v, "p_mun"] < 0.05) and not (inter.loc[v, "p_dep"] < 0.05)]
if cambian:
    print("Dejan de ser significativas al agrupar por departamento en vez de por municipio:", "; ".join(cambian))

**Lectura: qué soporta cada resultado frente a CDM-5 y ODM-3.**

- **¿Deforesta? — CDM-5 no se cumple, y se reporta así.** CDM-5 pide al menos 3 variables financieras o de producción con p < 0,05 corregido y errores por conglomerado espacial. Con errores por departamento (31 grupos) quedan
  2 de 10 variables de intervención (cacao como % del municipio, p = 0,004, y crédito ganadero, p = 0,012) y tras Holm solo 1: **cacao (% del municipio), OR = 1,41, IC 95 % 1,12-1,78, p Holm = 0,039**. Con errores por
  municipio serían 3, pero esa es la versión optimista (trata cuatro años de un mismo municipio y a municipios vecinos como independientes). Que el criterio no se cumpla no es un fallo del modelo: con 31 departamentos, diez variables
  correlacionadas y una corrección por comparaciones múltiples, es lo que los datos permiten afirmar.
- **¿Cuánto? — CDM-5 sí se cumple (4 variables tras Holm), con una advertencia.** Crédito ganadero (la tasa se multiplica por 0,58 por +1 desviación estándar), crédito a café (0,55), crédito a arroz (0,83) y bovinos por km² (1,79).
  Pero el modelo de magnitud tiene R² de 0,013 en 2025: son asociaciones dentro del entrenamiento, no capacidad de predecir cuánto.
- **Dirección.** Más crédito ganadero, cafetero y arrocero se asocia con menos riesgo y con tasas menores; más cacao y más bovinos, con más. Es **asociación, no causalidad**: el crédito llega sobre todo a zonas de agricultura
  consolidada, así que un OR menor que 1 puede reflejar *dónde* se presta, no un efecto protector del crédito. Además,
  `Fincas ganaderas por 100 km²` salió del modelo por VIF y quedó representada por el crédito ganadero (rho de Spearman = +0,81): parte de lo que dice el OR del crédito ganadero es presión ganadera.
- **Dos variables, dos pruebas.** El crédito ganadero es la variable de intervención que más aporta a la predicción (permutación: pasa Holm con p = 0,0001; efecto marginal de -5,0 puntos porcentuales por +1 desviación estándar, p = 0,012),
  pero no pasa Holm en la inferencia del logit (p = 0,108). El cacao hace lo contrario: pasa Holm en el logit y en la prueba conjunta de la actividad (p Holm < 0,001), pero no en la permutación por departamento. Por eso las dos
  actividades (ganadería y cacao) son las que hay que mirar, y por eso no conviene afirmar más de lo que cada prueba soporta.
- **No es un artefacto del calendario.** Al agregar efectos fijos de año ninguna conclusión cambia (los años mismos no son significativos), lo que responde al componente temporal que pide ODM-3.

#### Efectos marginales y pruebas conjuntas por actividad

El OR es un cociente de *odds*, difícil de comunicar. El **efecto marginal promedio** lo traduce a lo que le importa a un lector no
técnico: cuántos **puntos porcentuales** cambia la probabilidad de deforestar por cada +1 desviación estándar de la variable. Y la
**prueba de Wald conjunta** responde si **todas** las variables de una actividad son a la vez cero. Es la versión inferencial de la permutación
por bloques, y evita concluir que una actividad "no importa" solo porque sus variables, correlacionadas entre sí, salen
individualmente no significativas.

In [ ]:
efm = logit_dep.get_margeff(at="overall").summary_frame()
efm = efm.loc[VARS_LR, ["dy/dx", "Pr(>|z|)", "Conf. Int. Low", "Cont. Int. Hi."]]
efm.insert(1, "puntos_porcentuales", efm["dy/dx"] * 100)
efm.index = [NOMBRES[v] for v in efm.index]
print("Efecto marginal promedio: cambio en la probabilidad de deforestar por +1 desviación estándar (errores por departamento)")
display(efm.sort_values("dy/dx").round(4))


def wald(modelo_sm, cols):
    cols = [c for c in cols if c in Xsm.columns]
    if not cols:
        return {"n_variables": 0, "chi2": np.nan, "p_valor": np.nan}
    R = np.zeros((len(cols), len(Xsm.columns)))
    for i, c in enumerate(cols):
        R[i, Xsm.columns.get_loc(c)] = 1
    t = modelo_sm.wald_test(R, scalar=True)
    return {"n_variables": len(cols), "chi2": float(t.statistic), "p_valor": float(t.pvalue)}


pruebas = pd.concat({"bloque": pd.DataFrame({k: wald(logit_dep, v) for k, v in BLOQUES.items()}).T,
                     "actividad": pd.DataFrame({k: wald(logit_dep, v) for k, v in ACTIVIDADES.items()}).T})
pruebas["p_holm"] = np.nan
for nivel in ("bloque", "actividad"):
    ok = pruebas.loc[nivel].p_valor.notna()
    idx = pruebas.loc[nivel].index[ok]
    pruebas.loc[[(nivel, i) for i in idx], "p_holm"] = multipletests(pruebas.loc[nivel].loc[idx, "p_valor"].astype(float), method="holm")[1]
print("Pruebas de Wald conjuntas (errores por departamento); n_variables = las que quedaron tras el VIF:")
pruebas.round(4)

#### Sensibilidad: efectos fijos de año (la "estructura temporal" de ODM-3)

ODM-3 pide modelos "que incorporen la estructura espacial y temporal de los datos". La espacial entra con los errores agrupados por
departamento. La temporal: la deforestación y el crédito pueden moverse juntos por razones del año (una sequía, un cambio de política,
la inflación de los montos), no por una relación entre ellos. Agregar una variable indicadora por año quita ese componente común. Si las
asociaciones sobreviven, no son un artefacto del calendario.

In [ ]:
anios = pd.get_dummies(train.anio, prefix="anio", drop_first=True).astype(float)
Xfe = pd.concat([Xsm, anios], axis=1)
logit_fe = sm.Logit(y_train_clas, Xfe).fit(disp=0, cov_type="cluster", cov_kwds={"groups": cl_dep})
sens = pd.DataFrame({"OR sin efectos de año": np.exp(logit_dep.params), "p sin efectos de año": logit_dep.pvalues,
                     "OR con efectos de año": np.exp(logit_fe.params[Xsm.columns]), "p con efectos de año": logit_fe.pvalues[Xsm.columns]}
                    ).drop("const")
sens["cambia la significancia (5 %)"] = (sens["p sin efectos de año"] < 0.05) != (sens["p con efectos de año"] < 0.05)
sens.index = [NOMBRES[v] for v in sens.index]
cambian_fe = sens.index[sens["cambia la significancia (5 %)"]].tolist()
print("Variables cuya significancia (p < 0,05) cambia al agregar efectos fijos de año:", "; ".join(cambian_fe) or "ninguna")
print("Pruebas de los años en sí (p):", ", ".join(f"{c} = {p:.3f}" for c, p in logit_fe.pvalues.items() if c.startswith("anio_")))
sens.round(4)

#### ¿Cuánto? — efectos multiplicativos sobre la tasa (regresión Gamma con errores por departamento)

Para la magnitud, el modelo condicional (solo municipios-año que deforestan) se re-estima como GLM Gamma con enlace logarítmico,
sin regularización, con errores robustos por departamento. Cada coeficiente se lee como un efecto **multiplicativo** sobre la tasa
esperada de deforestación: `exp(β)` = cuántas veces cambia la tasa por +1 desviación estándar de la variable. Es la parte de
"magnitud, dirección e incertidumbre" de ODM-3 para el "¿cuánto?". Las advertencias de 5.3 siguen vigentes: la magnitud es
la parte menos predecible, y un solo municipio extremo pesa mucho.

In [ ]:
cond = en_deforesta
Xcond = Xsm[cond]
glm = sm.GLM(y_train_tasa[cond], Xcond, family=sm.families.Gamma(sm.families.links.Log())).fit(
    cov_type="cluster", cov_kwds={"groups": cl_dep[cond]})
mult = pd.DataFrame({"efecto_multiplicativo": np.exp(glm.params), "IC95_inf": np.exp(glm.conf_int()[0]),
                     "IC95_sup": np.exp(glm.conf_int()[1]), "p": glm.pvalues}).drop("const")
mult["p_holm"] = multipletests(mult.p, method="holm")[1]
mult["tipo"] = np.where(mult.index.isin(ESTRUCTURALES), "estructural", "intervención")
mult.index = [NOMBRES[v] for v in mult.index]
mult_interv = mult[mult.tipo == "intervención"]
cdm5_reg = {"n_dep": int((mult_interv.p < 0.05).sum()), "n_holm": int((mult_interv.p_holm < 0.05).sum()), "total": len(mult_interv),
            "nombres": mult_interv.index[mult_interv.p_holm < 0.05].tolist()}
print(f"Dado que deforesta, de las {cdm5_reg['total']} variables de intervención: {cdm5_reg['n_dep']} con p < 0,05 (errores por departamento), "
      f"{cdm5_reg['n_holm']} tras Holm: {', '.join(cdm5_reg['nombres']) or 'ninguna'}.")
mult.sort_values("efecto_multiplicativo", ascending=False).round(4)

### 7.3 Dirección del efecto

En una regresión lineal en el logit, la dirección es el signo del coeficiente: no hace falta SHAP (que, para un modelo lineal,
coincide exactamente con coeficiente × valor estandarizado). Lo que el gradient boosting y el bosque aleatorio obtienen con
SHAP y dependencia parcial, aquí se lee directamente. La tabla reúne las **tres cosas que pide ODM-3** para cada variable de
intervención —magnitud (OR y efecto marginal), dirección (signo) e incertidumbre (intervalo y valor p con errores por
departamento)— sin atribuir causalidad.

In [ ]:
resumen_dir = pd.DataFrame({
    "dirección": np.where(coefs.coef > 0, "sube el riesgo", "baja el riesgo"),
    "OR por +1 desv.": coefs.OR, "IC 95 % inf.": coefs.OR_IC95_inf_dep, "IC 95 % sup.": coefs.OR_IC95_sup_dep,
    "efecto marginal (pp)": (logit_dep.get_margeff(at="overall").summary_frame()["dy/dx"] * 100).reindex(coefs.index),
    "p (por departamento)": coefs.p_dep, "p Holm": coefs.p_holm_dep, "tipo": coefs.tipo})
resumen_dir["lectura"] = np.where(resumen_dir["p Holm"] < 0.05, "asociación clara", np.where(resumen_dir["p (por departamento)"] < 0.05, "asociación débil (no resiste Holm)", "sin evidencia"))
resumen_dir.index = [NOMBRES[v] for v in resumen_dir.index]
print("Variables de intervención, ordenadas por odds ratio:")
display(resumen_dir[resumen_dir.tipo == "intervención"].sort_values("OR por +1 desv.", ascending=False).round(3))
sube = resumen_dir[(resumen_dir.tipo == "intervención") & (resumen_dir.lectura == "asociación clara") & (resumen_dir["OR por +1 desv."] > 1)].index.tolist()
baja = resumen_dir[(resumen_dir.tipo == "intervención") & (resumen_dir.lectura == "asociación clara") & (resumen_dir["OR por +1 desv."] < 1)].index.tolist()
print("\nAsociación clara (resiste Holm) con MÁS riesgo:", "; ".join(sube) or "ninguna")
print("Asociación clara (resiste Holm) con MENOS riesgo:", "; ".join(baja) or "ninguna")
print("No es un efecto causal: el crédito puede ir a municipios que de entrada tienen menos presión sobre el bosque.")

### 7.4 Municipios de mayor riesgo (lista, sin necesitar el mapa)

La misma priorización de 6.3, municipio por municipio y con la razón concreta. En un modelo lineal, la contribución de cada variable a la
predicción de un municipio es **exacta**: `coeficiente × valor estandarizado` (el logit se descompone en una suma). Para los municipios-año de mayor
riesgo predicho en 2025 se muestran las tres variables que más empujaron la predicción y la **principal palanca de política** (la
variable de intervención con mayor contribución). Como en el gradient boosting, si la palanca no varía de un municipio a otro, lo que hay es un
arquetipo que se repite, no quince razones distintas.

In [ ]:
Z_test = pd.DataFrame(pre_f.transform(X_test), columns=VARS_LR, index=X_test.index)
contrib = Z_test * pesos.peso.reindex(VARS_LR)  # contribucion de cada variable al logit, frente al municipio promedio
nombres_politica_set = set(NOMBRES[v] for v in VARS_LR if v not in ESTRUCTURALES)

N_LISTA = 15
idx_riesgo = np.argsort(-prob_test)[:N_LISTA]
filas_lista = []
for pos in idx_riesgo:
    fila = test.iloc[pos]
    contribuciones = contrib.iloc[pos].rename(index=NOMBRES)
    top_razones = contribuciones.sort_values(ascending=False).head(3)
    razon_texto = "; ".join(f"{nombre} ({valor:+.2f})" for nombre, valor in top_razones.items())
    top_politica = contribuciones[contribuciones.index.isin(nombres_politica_set)].sort_values(ascending=False).head(1)
    filas_lista.append({
        "municipio": fila.nombre_municipio, "departamento": fila.nombre_departamento,
        "riesgo predicho": prob_test[pos], f"¿deforestó en {ANIO_PRUEBA}?": "Sí" if fila.deforesta == 1 else "No",
        f"tasa real {ANIO_PRUEBA} (%)": fila.tasa,
        "top 3 contribuciones (incluye estructurales)": razon_texto,
        "principal palanca de política (solo intervención)": f"{top_politica.index[0]} ({top_politica.iloc[0]:+.2f})",
    })
lista_riesgo = pd.DataFrame(filas_lista)
lista_riesgo.pipe(guardar_resultado, "regresion_logistica_lista_municipios_mayor_riesgo", indice=False)
n_palancas = lista_riesgo["principal palanca de política (solo intervención)"].str.split(" (", regex=False).str[0].nunique()
print(f"De los {N_LISTA} municipios-año de mayor riesgo predicho en {ANIO_PRUEBA}, "
      f"{(lista_riesgo[f'¿deforestó en {ANIO_PRUEBA}?'] == 'Sí').sum()} sí deforestaron realmente ese año.")
print(f"Palancas de política distintas que aparecen en el top {N_LISTA}: {n_palancas}"
      + (" — un solo arquetipo de riesgo que se repite." if n_palancas == 1 else "."))
lista_riesgo

### 7.5 Permutación, por bloques y por actividad (la misma de los otros dos modelos)

La **permutación** mide cuánto empeora el modelo en 2025 al barajar una variable (o un grupo de variables correlacionadas
a la vez). Es idéntica a la de `random_forest` y se calcula con las mismas 15 columnas de entrada: las dos que el VIF quitó tienen
importancia cero por construcción, y su información la lleva la variable que las representa. Es la **vara común** para comparar qué
pesa en cada técnica. Se reportan el valor p del gradient boosting (prueba t sobre las repeticiones; optimista), el del *bootstrap
por departamento* (el que se usa para contar) y el de Holm.

In [ ]:
perm_clas = permutacion_con_p(clasificador, X_test, y_test_clas, "clas", test.nombre_departamento, usadas=VARS_LR)
perm_reg = permutacion_con_p(regresor, X_test[en_deforesta_test], y_test_tasa_real, "reg",
                             test.nombre_departamento[en_deforesta_test], usadas=VARS_LR)
for t_ in (perm_clas, perm_reg):
    t_.loc[~t_.clave.isin(VARS_LR), "tipo"] = "excluida por VIF"

cols = ["variable", "tipo", "importancia_media", "p_valor", "p_territorio", "p_holm"]
print(f"¿deforesta? — permutación sobre {ANIO_PRUEBA} (caída del AUC), 8 primeras:")
display(perm_clas[cols].head(8).round(4))
print(f"Dado que deforesta, ¿cuánto? — permutación sobre {ANIO_PRUEBA} (caída del R²), 6 primeras:")
display(perm_reg[cols].head(6).round(4))

def contar(perm):
    inter = perm[(perm.tipo == "intervención") & (perm.importancia_media > 0)]
    return {"n_gb": int((inter.p_valor < 0.05).sum()), "n_terr": int((inter.p_territorio < 0.05).sum()),
            "n_holm": int((inter.p_holm < 0.05).sum()), "nombres": inter[inter.p_holm < 0.05].variable.tolist()}

perm_c, perm_r = contar(perm_clas), contar(perm_reg)
print(f"¿deforesta?: de las variables de intervención del modelo, {perm_c['n_gb']} con p < 0,05 (prueba t), {perm_c['n_terr']} con bootstrap por departamento, "
      f"{perm_c['n_holm']} tras Holm. ¿cuánto?: {perm_r['n_holm']} tras Holm.")

imp_bloques = {"bloques": permutar_bloques(clasificador, X_test, y_test_clas, BLOQUES, puntaje_auc),
               "actividades": permutar_bloques(clasificador, X_test, y_test_clas, ACTIVIDADES, puntaje_auc)}
graficar_bloques(imp_bloques, "AUC")
print("¿deforesta? — por bloque:")
display(imp_bloques["bloques"].round(4))
print("¿deforesta? — por actividad productiva:")
display(imp_bloques["actividades"].round(4))
X_test_def = X_test[en_deforesta_test]
imp_bloques_reg = {"bloques": permutar_bloques(regresor, X_test_def, y_test_tasa_real, BLOQUES, puntaje_r2),
                   "actividades": permutar_bloques(regresor, X_test_def, y_test_tasa_real, ACTIVIDADES, puntaje_r2)}
print("Dado que deforesta, ¿cuánto? — por actividad (caída del R²; muy ruidosa por Pedraza y por el poco R² de base):")
imp_bloques_reg["actividades"].round(4)

In [ ]:
act = imp_bloques["actividades"]
sig = act[(act.caida > 0) & (act.p_valor < 0.05)]
print("Actividades cuya permutación empeora el AUC de forma significativa (p < 0,05):",
      ", ".join(f"{i.split(' (')[0]} ({r.caida:.3f})" for i, r in sig.iterrows()) or "ninguna")
print(f"La actividad que más aporta es {act.index[0].split(' (')[0]} y la que menos, {act.index[-1].split(' (')[0]}.")
pw = pruebas.loc["actividad"]
print("Prueba conjunta de Wald (Holm) por actividad:", "; ".join(f"{i.split(' (')[0]} p = {r.p_holm:.3f}" for i, r in pw.iterrows() if pd.notna(r.p_holm)))

## 8. Lo que pasa al notebook 16

Los resultados se ven en este notebook. En BigQuery solo quedan las tablas de datos, así que lo único que se
guarda es lo que compara el notebook 16, en tablas de paso de `DATASET_RESULTADOS` que el 16 borra al terminar:
las métricas (con el mismo esquema que las del gradient boosting: `modelo`, `métrica`, `prueba_temporal_2025`,
`prueba_departamentos_fuera`, `brecha_%`), los hiperparámetros, la importancia por permutación, la lista de
municipios de mayor riesgo, las especificaciones y las predicciones del año de prueba, más los coeficientes y los efectos multiplicativos que usa el criterio CDM-5. Los modelos
entrenados no se guardan: correr el notebook los reconstruye iguales.

In [ ]:
def fila(modelo, metrica, temporal, deptos, brecha=np.nan):
    return {"modelo": modelo, "métrica": metrica, "prueba_temporal_2025": temporal,
            "prueba_departamentos_fuera": deptos, "brecha_%": brecha}

CLAS, REG = "Regresión logística — ¿deforesta?", "Regresión logística — ¿cuánto?"
metricas = pd.DataFrame([
    fila(CLAS, "exactitud balanceada", ba_test, ba_departamentos, brecha_clas),
    fila(CLAS, "AUC", m_clas_test["auc"], m_clas_oof["auc"]),
    fila(CLAS, "precisión", m_clas_test["precision"], m_clas_oof["precision"]),
    fila(CLAS, "recall / sensibilidad", m_clas_test["recall"], m_clas_oof["recall"]),
    fila(CLAS, "especificidad", m_clas_test["especificidad"], m_clas_oof["especificidad"]),
    fila(CLAS, "F1", m_clas_test["f1"], m_clas_oof["f1"]),
    fila(CLAS, "precisión promedio (AP, área bajo precisión-recall)", m_clas_test["ap"], m_clas_oof["ap"]),
    fila(REG, "R²", r2_test, r2_departamentos, brecha_reg),
    fila(REG, "RMSE", rmse_test, rmse_departamentos),
    fila(REG, "error absoluto promedio (MAE)", mae_test, mae_departamentos),
    fila(REG, "error absoluto mediano (MedAE)", medae_test, medae_departamentos),
    fila(REG, "R² en escala log (diagnóstico robusto)", m_reg_test["r2_log"], m_reg_oof["r2_log"]),
    fila(REG, "correlación de Spearman (diagnóstico robusto)", m_reg_test["spearman"], m_reg_oof["spearman"]),
    fila(REG, "R² excluyendo a Pedraza (diagnóstico, 5.3)", r2_sin_pedraza, np.nan, brecha_sin_pedraza),
    fila(REG, "R² una sola regresión sin separar, solo donde sí deforestó (diagnóstico, 5.2)",
         r2_score(y_test_tasa_real, pred_directo_def), np.nan),
])
metricas.pipe(guardar_resultado, "regresion_logistica_metricas", indice=False)
metricas

pd.DataFrame({"clasificación": {k: str(v) for k, v in params_clas.items()},
              "regresión": {k: str(v) for k, v in params_reg.items()}}).pipe(guardar_resultado, "regresion_logistica_hiperparametros", indice=True)
perm_clas.pipe(guardar_resultado, "regresion_logistica_importancia_permutacion_clasificacion", indice=False)
coefs_tabla.pipe(guardar_resultado, "regresion_logistica_coeficientes_inferencia", indice=True)
mult.pipe(guardar_resultado, "regresion_logistica_efectos_multiplicativos_magnitud", indice=True)
especificaciones_tabla.pipe(guardar_resultado, "regresion_logistica_especificaciones", indice=True)
test.assign(prob_deforesta=prob_test, pred_deforesta=pred_test).assign(
    tasa_predicha=pd.Series(regresor.predict(X_test), index=test.index))[
    ["cod_dane", "nombre_municipio", "nombre_departamento", "anio", "deforesta", "tasa", "prob_deforesta", "pred_deforesta",
     "tasa_predicha"]].pipe(guardar_resultado, "regresion_logistica_predicciones_prueba", indice=False)

## 9. Síntesis: qué responde este notebook de los objetivos del documento

Cada fila de la tabla final se **calcula** con los resultados de este notebook (no está escrita a mano), de modo que no puede contradecir a las
secciones anteriores. Está pensada para la sustentación (coherencia entre presentación y documento) y para que las tres técnicas se lean con la
misma plantilla. Antes, dos lecturas que el gradient boosting todavía no hace.

### 9.1 ¿Dónde coinciden alto crédito y alto riesgo? (OBJ-N1 y ODM-2)

Se miran dos formas de "alto crédito" porque cuentan historias distintas: el **volumen** (cuánta plata llega al municipio) y la **intensidad** (cuánta por km²).
Los municipios grandes de la frontera amazónica pueden recibir mucho en volumen y casi nada por km².

In [ ]:
# OBJ-N1 exige identificar "al menos un municipio o cadena donde coincidan alto financiamiento agropecuario
# y alto riesgo de deforestacion" (ODM-2). Con las probabilidades "limpias" de la seccion 6 (cada municipio
# predicho por un modelo que nunca vio su departamento) se promedia el riesgo de cada municipio en 2021-2025 y
# se cruza con su financiamiento de dos maneras, que cuentan cosas distintas:
#   - VOLUMEN: credito de produccion acumulado 2021-2025 (millones de pesos). Responde "¿cuanta plata llega?".
#   - INTENSIDAD: credito por km² (la forma en que lo usan los modelos). Responde "¿que tan densa es?".
CADENAS_CRED = {"credito_ganaderia_produccion_mm": "Ganadería", "credito_cafe_produccion_mm": "Café",
                "credito_cacao_produccion_mm": "Cacao", "credito_palma_produccion_mm": "Palma",
                "credito_arroz_produccion_mm": "Arroz"}
# Las columnas de texto se pasan a str: si vienen como `category`, groupby arma TODAS las combinaciones posibles
# (producto cartesiano, millones de "municipios" vacios) y los cortes por cuartiles quedan en cero.
cred_abs = anual[["cod_dane", "anio", "credito_produccion_mm"] + list(CADENAS_CRED)].fillna(0).astype({"cod_dane": str})
por_mun = (m[["cod_dane", "anio", "nombre_municipio", "nombre_departamento", "tasa", "def_ideam_ha", "cred"]]
           .astype({"cod_dane": str, "nombre_municipio": str, "nombre_departamento": str})
           .assign(riesgo=prob_clas_oof.values).merge(cred_abs, on=["cod_dane", "anio"], how="left")
           .groupby(["cod_dane", "nombre_municipio", "nombre_departamento"], observed=True)
           .agg(riesgo_medio=("riesgo", "mean"), ha_total=("def_ideam_ha", "sum"), credito_total_mm=("credito_produccion_mm", "sum"),
                credito_por_km2=("cred", "mean"), **{c: (c, "sum") for c in CADENAS_CRED}).reset_index())
assert len(por_mun) <= m.cod_dane.nunique(), "por_mun debe tener un renglon por municipio"
por_mun["cadena_dominante"] = por_mun[list(CADENAS_CRED)].idxmax(axis=1).map(CADENAS_CRED)
corte_riesgo = por_mun.riesgo_medio.quantile(0.75)
por_mun["alto_riesgo"] = por_mun.riesgo_medio >= corte_riesgo
resumen = {}
for nombre, col in [("volumen", "credito_total_mm"), ("intensidad", "credito_por_km2")]:
    corte = por_mun[col].quantile(0.75)
    por_mun[f"alto_credito_{nombre}"] = por_mun[col] >= corte
    sel = por_mun[por_mun.alto_riesgo & por_mun[f"alto_credito_{nombre}"]]
    resumen[nombre] = dict(corte=corte, n=len(sel), pct_ha=100 * sel.ha_total.sum() / por_mun.ha_total.sum())
    print(f"{nombre.upper()}: {len(sel)} de {len(por_mun)} municipios están en el cuarto superior de riesgo predicho "
          f"(≥ {corte_riesgo:.2f}) Y en el cuarto superior de crédito por {nombre} (≥ {corte:,.1f}); concentran el "
          f"{resumen[nombre]['pct_ha']:.1f} % de las hectáreas deforestadas del IDEAM {ANIO_INICIO}-{ANIO_PRUEBA}.")
coinciden = por_mun[por_mun.alto_riesgo & por_mun.alto_credito_volumen].sort_values("ha_total", ascending=False)
cruce = (por_mun.groupby("cadena_dominante", observed=True)
         .agg(municipios=("cod_dane", "size"), alto_riesgo=("alto_riesgo", "sum"),
              alto_riesgo_y_alto_volumen=("alto_credito_volumen", lambda s: int((s & por_mun.loc[s.index, "alto_riesgo"]).sum())),
              ha_deforestadas=("ha_total", "sum")).sort_values("ha_deforestadas", ascending=False))
print("\nPor cadena dominante del municipio (la de mayor crédito acumulado):")
display(cruce.round(0).astype(int))
print("Municipios con alto riesgo y alto volumen de crédito, por hectáreas deforestadas:")
coinciden[["nombre_municipio", "nombre_departamento", "riesgo_medio", "credito_total_mm", "credito_por_km2",
           "cadena_dominante", "ha_total"]].head(10).round(2)

**Lectura: qué responde el cruce alto crédito-alto riesgo (OBJ-N1 y ODM-2).**

- **Hay una coincidencia concreta, y es de volumen, no de intensidad.** 57 de 639 municipios están a la vez en el cuarto superior de riesgo y en el cuarto superior de crédito por volumen, y concentran el 59,7 % de las hectáreas
  deforestadas 2021-2025. Si el crédito se mide por km², solo coinciden 2 municipios (0,4 % de las hectáreas). Dicho de otra forma: la plata llega en monto grande a los municipios grandes de la frontera, pero delgada por km².
- **La cadena que domina el cruce es la ganadería** (37 de los 57), seguida por palma (7, p. ej. Mapiripán), arroz (6), café (5) y cacao (2). Los municipios con más hectáreas son Cartagena del Chairá, San Vicente del Caguán (Caquetá),
  Mapiripán y La Macarena (Meta).
- **Para el objetivo**, esto da lo que OBJ-N1 pide (al menos un municipio o cadena donde coinciden alto financiamiento y alto riesgo: ganadería en el arco Caquetá-Meta) y orienta ODM-2: el foco no es "dar menos crédito", sino qué condiciones
  tiene el crédito de ganadería y palma en esos municipios. Sigue siendo una coincidencia descriptiva: no prueba que el crédito cause la deforestación ni que retirarlo la reduzca.

### 9.2 Reproducibilidad (CDM-6)

Una prueba explícita en lugar de una afirmación: se vuelve a entrenar el clasificador y el regresor con los mismos hiperparámetros y se comprueba que las probabilidades y las
tasas de 2025 son **idénticas**. La regresión regularizada es determinista, así que aquí sí se espera igualdad exacta.

In [ ]:
rep_clas = modelo_clas(params_clas).fit(X_train, y_train_clas)
rep_reg = modelo_reg(params_reg).fit(X_train[en_deforesta], y_train_tasa[en_deforesta])
dif_clas = float(np.abs(rep_clas.predict_proba(X_test)[:, 1] - prob_test).max())
dif_reg = float(np.abs(rep_reg.predict(X_test) - regresor.predict(X_test)).max())
reproducible = dif_clas < 1e-12 and dif_reg < 1e-12
print(f"Reentrenar da la misma probabilidad (diferencia máxima {dif_clas:.1e}) y la misma tasa (diferencia máxima {dif_reg:.1e}) "
      f"en los 639 municipios de {ANIO_PRUEBA}: {'SÍ' if reproducible else 'NO'}.")

### 9.3 Tabla de síntesis por objetivo

In [ ]:
nombres_credito_lr = {NOMBRES[v] for v in CREDITO}
cred_clas_ok = [NOMBRES[v] for v in coefs.index if v in CREDITO and coefs.loc[v, "p_holm_dep"] < 0.05]
cred_reg_ok = mult[(mult.index.isin(nombres_credito_lr)) & (mult.p_holm < 0.05)]
texto_credito_lr = ("¿deforesta?: " + (", ".join(cred_clas_ok) + " resiste Holm. " if cred_clas_ok else
                    f"ninguna línea de crédito resiste Holm (la de mayor peso, {NOMBRES[top_credito_lr]}, tiene p = {coefs.loc[top_credito_lr, 'p_dep']:.3f} sin corregir y {coefs.loc[top_credito_lr, 'p_holm_dep']:.3f} con Holm). ")
                    + "¿cuánto?: " + ("; ".join(f"{i} (la tasa se multiplica por {r.efecto_multiplicativo:.2f} por +1 desv.)" for i, r in cred_reg_ok.iterrows())
                                      + " resisten Holm." if len(cred_reg_ok) else "ninguna línea de crédito resiste Holm."))
a_est, b_est = especificaciones_tabla.loc["A. Estructural", "AUC"], especificaciones_tabla.loc["B. Principal (tras VIF)", "AUC"]
cob = cobertura_oof[30]
dep_top = por_depto.index[:3].tolist()
estado_brecha = lambda b: "cumple" if (pd.notna(b) and abs(b) <= 15) else "no cumple"
reg_brecha_txt = ((f"R² {r2_train:.2f} → {r2_test:.2f} (brecha {brecha_reg:.0f} %, {estado_brecha(brecha_reg)}); sin Pedraza {r2_sin_pedraza:.2f} "
                   f"(brecha {brecha_sin_pedraza:.0f} %, {estado_brecha(brecha_sin_pedraza)})") if r2_train > 0
                  else f"R² de entrenamiento no positivo ({r2_train:.2f}): la brecha relativa no es interpretable; R² de prueba {r2_test:.2f}")

sintesis = pd.DataFrame([
    {"objetivo / criterio": "OBJ-N1 · priorización territorial",
     "evidencia en este notebook": (f"AUC {auc_test:.3f} en {ANIO_PRUEBA} y {m_clas_oof['auc']:.3f} con departamentos fuera; revisando el 30 % de "
                                    f"municipios-año de mayor riesgo se cubre el {cob['pct_municipios']:.0f} % de los que deforestan y el "
                                    f"{cob['pct_ha']:.0f} % de las hectáreas. Departamentos de mayor riesgo: {', '.join(dep_top)}. "
                                    f"Alto riesgo y alto volumen de crédito coinciden en {resumen['volumen']['n']} municipios ({resumen['volumen']['pct_ha']:.0f} % de las hectáreas), "
                                    f"pero solo en {resumen['intensidad']['n']} si el crédito se mide por km² (9.1)."),
     "estado": "evidencia técnica aportada"},
    {"objetivo / criterio": "CEN-2 · validación por actores externos",
     "evidencia en este notebook": "No se mide aquí: requiere la validación de actores del mercado objetivo. Este notebook aporta la cifra verificable (curva de ganancia, ranking) que se les presenta.",
     "estado": "no aplica en el notebook"},
    {"objetivo / criterio": "OBJ-N2 / CEN-1 · autosostenibilidad",
     "evidencia en este notebook": "Ningún modelo cierra una venta por sí mismo; aporta el activo reproducible (notebook, modelo, hallazgos) que el Observatorio puede mostrar.",
     "estado": "no aplica en el notebook"},
    {"objetivo / criterio": "OBJ-N3 / CEN-3 / CDM-6 · activo reutilizable y reproducible",
     "evidencia en este notebook": f"Semilla fija: correr el notebook da los mismos resultados; reentrenar da exactamente las mismas predicciones: {'sí' if reproducible else 'no'}.",
     "estado": "cumple" if reproducible else "revisar"},
    {"objetivo / criterio": "ODM-2 · cruce de financiamiento y producción",
     "evidencia en este notebook": (f"Por actividad (permutación del AUC), la que más aporta es {act.index[0].split(' (')[0]} y la que menos, {act.index[-1].split(' (')[0]}. "
                                    + texto_credito_lr),
     "estado": "evidencia técnica aportada"},
    {"objetivo / criterio": "ODM-3 / CDM-5 · variables con asociación significativa",
     "evidencia en este notebook": (f"¿deforesta?: de {cdm5['total']} variables de intervención, {cdm5['n_mun']} con p < 0,05 con errores por municipio, {cdm5['n_dep']} por departamento y "
                                    f"{cdm5['n_holm']} tras Holm ({', '.join(cdm5['nombres']) or 'ninguna'}). ¿cuánto?: {cdm5_reg['n_holm']} tras Holm. "
                                    f"Sensibilidad con efectos fijos de año: {'cambia la significancia de ' + '; '.join(cambian_fe) if cambian_fe else 'no cambia ninguna conclusión'}."),
     "estado": (("cumple (≥ 3)" if cdm5["n_holm"] >= 3 else "no cumple (< 3)") + " en ¿deforesta?; "
                + ("cumple" if cdm5_reg["n_holm"] >= 3 else "no cumple") + f" en ¿cuánto? ({cdm5_reg['n_holm']})")},
    {"objetivo / criterio": "CDM-7 · brecha entrenamiento-prueba ≤ 15 %",
     "evidencia en este notebook": (f"¿deforesta?: entrenamiento {brecha_clas:.0f} %, validación por departamentos {brecha_clas_cv:.0f} %. ¿cuánto?: {reg_brecha_txt}."),
     "estado": "cumple (¿deforesta?)" if abs(brecha_clas) <= 15 and abs(brecha_clas_cv) <= 15 else "revisar"},
    {"objetivo / criterio": "REQ-04 · modelos interpretables, sin cajas negras",
     "evidencia en este notebook": "Coeficientes con signo, odds ratios con intervalo, efectos marginales, pruebas conjuntas por actividad y contribución exacta de cada variable a la predicción de cada municipio.",
     "estado": "cumple"},
    {"objetivo / criterio": "Lectura global para la sustentación",
     "evidencia en este notebook": (f"(1) Priorizar sí se puede: AUC {auc_test:.2f} en {ANIO_PRUEBA} y {m_clas_oof['auc']:.2f} en departamentos nuevos, con un modelo lineal que casi no memoriza (brecha {brecha_clas:.0f} %). "
                                    f"(2) La estructura pesa {pesos.groupby('tipo')['participacion_%'].sum()['estructural']:.0f} % del peso total de los coeficientes; bosque y área solos dan AUC {a_est:.3f} y el modelo completo {b_est:.3f} ({b_est - a_est:+.3f}). "
                                    f"(3) Con errores por departamento, la asociación financiera/productiva es frágil en ¿deforesta? ({cdm5['n_holm']} resiste Holm) y más clara en ¿cuánto? ({cdm5_reg['n_holm']}). "
                                    f"(4) El alto crédito coincide con el alto riesgo por volumen ({resumen['volumen']['n']} municipios), no por intensidad ({resumen['intensidad']['n']})."),
     "estado": "resumen"},
])
pd.set_option("display.max_colwidth", 400)
sintesis

## Próximos pasos y límites

- **Los tres modelos hablan el mismo idioma.** Las tres técnicas comparten muestra, variables, partición, pliegues, umbral, regla de búsqueda (grilla exhaustiva con opción de ponderar clases) y métricas, y guardan sus
  resultados con el mismo esquema. La comparación está en `16_modelos_comparacion`: este notebook no decide cuál modelo es mejor, solo aporta lo que la logística hace y los otros no.
- **Lo que solo tiene este notebook.** La inferencia con errores robustos por departamento (OR con intervalo, efectos marginales), las pruebas de Wald conjuntas por actividad, la sensibilidad con efectos fijos de año, la
  especificación sin podar y la descomposición exacta de la predicción de cada municipio. El gradient boosting y el random forest ya comparten la permutación con bootstrap por departamento, los bloques y las especificaciones A-D.
- **Qué sí soporta este modelo (y para qué se usa).** Priorizar territorios con la estructura que ya se conoce (bosque y tamaño), en departamentos nuevos y con una brecha de 0,7 %; y leer asociaciones con signo, magnitud e
  incertidumbre. Es la referencia interpretable: si un ensamble no le gana por un margen que justifique perder los odds ratios, la elección se hace por explicabilidad.
- **Qué no soporta.** Cuantificar hectáreas ("¿cuánto?": R² de 0,013, o 0,112 sin Pedraza); afirmar causalidad; ni decir que el crédito "protege" el bosque. CDM-5 no se cumple en ¿deforesta? con la prueba exigente (1 variable) y sí en
  ¿cuánto? (4), con la advertencia de que la magnitud casi no se predice.
- **La magnitud sigue siendo el punto débil**, igual que en los árboles: Pedraza y unos pocos municipios del corredor seco del Caribe concentran el error. Un modelo lineal en el logaritmo de la tasa, o una variable que distinga los
  municipios de bosque casi nulo, serían los siguientes pasos.
- **Supuestos que se vigilaron.** Independencia (errores y validación por departamento, año completo reservado), linealidad en el logit (log(1+x)), multicolinealidad (VIF < 5) y desbalance (`class_weight` elegido por validación).
  La linealidad en el logit no se contrastó con un término cuadrático; la dependencia parcial de `random_forest` es la lectura que mostraría una relación no monótona.
- **Reproducibilidad.** Semilla fija y modelo determinista: reentrenar da exactamente las mismas predicciones (9.2). Las cifras de las lecturas escritas arriba son las de la corrida del 20/11/2026; si se vuelve a correr con otras versiones de
  las librerías, hay que verificarlas contra las salidas de cada celda.